# LeetCode 460 — LFU Cache bằng Python (Hard)

Notebook này giảng nhập môn một bài **thiết kế class** (design) kinh điển trong phỏng vấn:

```python
class LFUCache:

    def __init__(self, capacity: int):


    def get(self, key: int) -> int:


    def put(self, key: int, value: int) -> None:



# Your LFUCache object will be instantiated and called as such:
# obj = LFUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)
```

Mục tiêu của bài học:

1. Hiểu **cache** là gì, thế nào là **chính sách loại bỏ** (eviction policy), và sự khác nhau giữa **LRU** (Least Recently Used — lâu nhất chưa dùng) và **LFU** (Least Frequently Used — ít được dùng nhất).
2. Đọc được input dạng chuỗi lệnh `["LFUCache","put","get",...]`, `[[2],[1,1],[1],...]` và tự viết hàm `run_commands` để chạy test giống hệt LeetCode.
3. Ôn lại **LRU Cache (#146)** bằng `OrderedDict` từ con số 0: `move_to_end`, `popitem(last=False)`.
4. Đi từng bước qua 4 cách làm, mỗi bước chỉ rõ cách cũ vướng ở đâu:
   - **Cách 1** — `dict` + quét tìm key ít dùng nhất (kèm thời điểm dùng gần nhất), `O(capacity)` mỗi lần loại. Dễ hiểu nhất, dùng làm **trọng tài** (referee) khi test.
   - **Cách 2** — **heap** `(freq, time, key)` + **lazy deletion** (xóa lười), `O(log)` mỗi thao tác.
   - **Cách 3 (cách chính)** — `key → (value, freq)` + `freq → OrderedDict` + biến `min_freq`, **`O(1)` trung bình** cho cả `get` và `put`.
   - **Cách 4 (tùy chọn)** — tự cài **danh sách liên kết đôi** (doubly linked list) thay cho `OrderedDict`.
5. Trace kỹ ví dụ của đề bằng bảng trạng thái các **tầng tần suất**, hiểu vì sao `min_freq` chỉ cần cập nhật theo 2 quy tắc rất đơn giản.
6. Tự kiểm chứng bằng test chéo ngẫu nhiên có `random.seed`, tránh các lỗi kinh điển và nhận ra pattern "nhóm theo tần suất" ở các bài khác.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Hãy thiết kế và cài đặt một cấu trúc dữ liệu cho **LFU cache** (Least Frequently Used — loại bỏ phần tử *ít được dùng nhất*).
>
> Cài đặt class `LFUCache`:
>
> - `LFUCache(int capacity)` khởi tạo object với **sức chứa** `capacity`.
> - `int get(int key)` trả về giá trị của `key` nếu `key` có trong cache; ngược lại trả về `-1`.
> - `void put(int key, int value)` cập nhật giá trị của `key` nếu đã có, hoặc chèn `key` nếu chưa có. Khi cache **đã đầy** (đạt `capacity`), phải **loại bỏ key ít được dùng nhất** *trước khi* chèn phần tử mới. Trong bài này, khi **hòa** (hai hay nhiều key có cùng tần suất), key **lâu nhất chưa được dùng** (least recently used) sẽ bị loại.
>
> Để xác định key ít được dùng nhất, mỗi key trong cache có một **bộ đếm số lần dùng** (use counter). Key có bộ đếm nhỏ nhất là key ít được dùng nhất.
>
> Khi một key được chèn lần đầu vào cache, bộ đếm của nó được đặt bằng `1` (do chính lệnh `put` đó). Bộ đếm của một key trong cache tăng thêm 1 mỗi khi có lệnh `get` **hoặc** `put` được gọi trên key đó.
>
> Các hàm `get` và `put` đều phải chạy với độ phức tạp **`O(1)` trung bình**.

**Example 1**

```text
Input
["LFUCache", "put", "put", "get", "put", "get", "get", "put", "get", "get", "get"]
[[2], [1, 1], [2, 2], [1], [3, 3], [2], [3], [4, 4], [1], [3], [4]]
Output
[null, null, null, 1, null, -1, 3, null, -1, 3, 4]

Explanation
// cnt(x) = bộ đếm số lần dùng của key x
// cache=[] cho biết thứ tự dùng gần nhất để phân xử khi hòa (phần tử bên trái là mới dùng nhất)
LFUCache lfu = new LFUCache(2);
lfu.put(1, 1);   // cache=[1,_], cnt(1)=1
lfu.put(2, 2);   // cache=[2,1], cnt(2)=1, cnt(1)=1
lfu.get(1);      // trả về 1
                 // cache=[1,2], cnt(2)=1, cnt(1)=2
lfu.put(3, 3);   // 2 là key LFU vì cnt(2)=1 nhỏ nhất, loại 2.
                 // cache=[3,1], cnt(3)=1, cnt(1)=2
lfu.get(2);      // trả về -1 (không tìm thấy)
lfu.get(3);      // trả về 3
                 // cache=[3,1], cnt(3)=2, cnt(1)=2
lfu.put(4, 4);   // 1 và 3 cùng cnt, nhưng 1 lâu chưa dùng hơn (LRU), loại 1.
                 // cache=[4,3], cnt(4)=1, cnt(3)=2
lfu.get(1);      // trả về -1 (không tìm thấy)
lfu.get(3);      // trả về 3
                 // cache=[3,4], cnt(4)=1, cnt(3)=3
lfu.get(4);      // trả về 4
                 // cache=[4,3], cnt(4)=2, cnt(3)=3
```

**Constraints** (bản đề hiện hành trên LeetCode)

- `1 <= capacity <= 10^4`
- `0 <= key <= 10^5`
- `0 <= value <= 10^9`
- Có tối đa `2 * 10^5` lời gọi `get` và `put`.

> **Ghi chú về `capacity`.** Bản đề hiện hành ghi `1 <= capacity`, tức cache luôn chứa được ít nhất 1 key. Các bản đề **cũ hơn** của bài này từng ghi `0 <= capacity` (có test `capacity = 0`, khi đó mọi `put` đều vô tác dụng). Vì vậy, trong code nộp ta vẫn giữ **một dòng phòng thủ** `if self.capacity == 0: return` — rẻ, không hại gì, và không bao giờ bị "Runtime Error" nếu gặp bộ test cũ. Ở mục 12 ta sẽ thấy code thiếu dòng này nổ lỗi ra sao.

### Ký hiệu dùng suốt notebook

- `c` = `capacity` (sức chứa, `c <= 10^4`).
- `q` = tổng số lời gọi `get` + `put` (`q <= 2·10^5`).
- `freq` (hay `cnt`) = bộ đếm số lần dùng của một key.

### 1.1 Đọc input / output của bài "thiết kế class"

LeetCode mô tả một **kịch bản gọi hàm** bằng 2 list song song: list thứ nhất là **tên lệnh**, list thứ hai là **tham số** của lệnh tương ứng. Lệnh đầu tiên luôn là tên class (gọi constructor).

| Vị trí | Tên lệnh | Tham số | Nghĩa là | Output |
|---:|---|---|---|---|
| 0 | `"LFUCache"` | `[2]` | `lfu = LFUCache(2)` | `null` |
| 1 | `"put"` | `[1, 1]` | `lfu.put(1, 1)` | `null` (hàm `void` không trả gì) |
| 2 | `"put"` | `[2, 2]` | `lfu.put(2, 2)` | `null` |
| 3 | `"get"` | `[1]` | `lfu.get(1)` | `1` |
| 4 | `"put"` | `[3, 3]` | `lfu.put(3, 3)` — cache đầy, phải loại 1 key | `null` |
| 5 | `"get"` | `[2]` | `lfu.get(2)` — key 2 đã bị loại | `-1` |
| … | … | … | … | … |

- `null` là cách Java / JSON viết "không có giá trị"; trong Python tương ứng là `None`.
- Đừng bỏ qua câu "bộ đếm tăng khi `get` **hoặc** `put` được gọi trên key đó": `put(k, v)` với `k` **đã có** cũng là một lần **dùng** `k`, nên `cnt(k)` tăng. Đây là chỗ rất nhiều bạn làm sai.

### Vì sao bài này là Hard?

Nếu chỉ cần **đúng** thì không khó: lưu mọi key kèm bộ đếm, khi cần loại thì quét tìm key có bộ đếm nhỏ nhất. Cái khó là đề đòi **`O(1)` trung bình** cho **cả hai** thao tác, trong khi ta phải trả lời câu hỏi "trong tất cả các key, key nào có `freq` nhỏ nhất, và nếu hòa thì key nào lâu chưa dùng nhất?" — một câu hỏi nghe như phải quét toàn bộ. Notebook sẽ đi từng bước từ cách quét `O(c)` → heap `O(log)` → cấu trúc "các tầng tần suất" `O(1)`.

Cell dưới đây chép ví dụ của đề vào biến để dùng lại suốt notebook.

In [1]:
from typing import List, Optional

commands_ex1 = ["LFUCache", "put", "put", "get", "put", "get", "get", "put", "get", "get", "get"]
args_ex1     = [[2], [1, 1], [2, 2], [1], [3, 3], [2], [3], [4, 4], [1], [3], [4]]
expected_ex1 = [None, None, None, 1, None, -1, 3, None, -1, 3, 4]

for i, (cmd, a, e) in enumerate(zip(commands_ex1, args_ex1, expected_ex1)):
    print(f"{i:>2}  {cmd:<9} {str(a):<7} -> {e}")

 0  LFUCache  [2]     -> None
 1  put       [1, 1]  -> None
 2  put       [2, 2]  -> None
 3  get       [1]     -> 1
 4  put       [3, 3]  -> None
 5  get       [2]     -> -1
 6  get       [3]     -> 3
 7  put       [4, 4]  -> None
 8  get       [1]     -> -1
 9  get       [3]     -> 3
10  get       [4]     -> 4


## 2. Nền tảng

### 2.1 Cache là gì? Vì sao phải "loại bỏ"?

Hình dung bàn học của bạn chỉ để vừa **2 quyển sách** (đó là `capacity = 2`), còn lại để trên giá sách ở phòng bên cạnh.

- Cần quyển nào mà nó **đang ở trên bàn** → lấy ngay, rất nhanh. Gọi là **cache hit** (trúng cache).
- Cần quyển không có trên bàn → phải đi lấy, chậm. Gọi là **cache miss** (trượt cache). Trong bài này, `get` trượt thì trả `-1`.
- Bàn đã đủ 2 quyển mà muốn đặt quyển thứ 3 → **phải cất bớt một quyển** về giá. Quy tắc chọn quyển nào để cất gọi là **eviction policy** (chính sách loại bỏ).

Ba chính sách hay gặp:

| Chính sách | Loại phần tử nào? | Đời thường |
|---|---|---|
| **FIFO** (First In First Out) | Phần tử vào **sớm nhất** | Cất quyển đặt lên bàn đầu tiên |
| **LRU** (Least Recently Used) | Phần tử **lâu nhất chưa được dùng** | Cất quyển lâu rồi không mở |
| **LFU** (Least Frequently Used) | Phần tử **ít lần được dùng nhất**; hòa thì loại cái lâu chưa dùng nhất | Cất quyển ít khi đụng tới |

LFU cần nhớ **hai thông tin** cho mỗi key:

1. `cnt` — đã được dùng bao nhiêu lần (để so "ít dùng").
2. `last` — lần dùng gần nhất là **lúc nào** (để phân xử khi hòa `cnt`).

"Lúc nào" không cần giờ thật. Ta dùng một **đồng hồ logic** (logical clock): một biến đếm tăng 1 mỗi lần có thao tác dùng key. Số càng lớn = dùng càng gần đây. Cell dưới đếm hai thông tin này cho một dãy truy cập nhỏ.

In [2]:
accesses = ["A", "B", "A", "C", "A", "B"]    # thứ tự các lần dùng

cnt = {}    # key -> số lần dùng
last = {}   # key -> thời điểm dùng gần nhất (đồng hồ logic)
for t, x in enumerate(accesses, start=1):    # enumerate(..., start=1): t chạy 1, 2, 3, ...
    cnt[x] = cnt.get(x, 0) + 1                # dict.get(key, mặc định): chưa có thì coi là 0
    last[x] = t

print("cnt  =", cnt)
print("last =", last)
print("LRU muốn loại :", min(last, key=lambda k: last[k]), "(last nhỏ nhất)")
print("LFU muốn loại :", min(cnt, key=lambda k: (cnt[k], last[k])), "(cnt nhỏ nhất, hòa thì last nhỏ nhất)")

cnt  = {'A': 3, 'B': 2, 'C': 1}
last = {'A': 5, 'B': 6, 'C': 4}
LRU muốn loại : C (last nhỏ nhất)
LFU muốn loại : C (cnt nhỏ nhất, hòa thì last nhỏ nhất)


Đọc output:

- `A` dùng 3 lần, `B` 2 lần, `C` 1 lần. Lần dùng gần nhất: `A` lúc 5, `B` lúc 6, `C` lúc 4.
- **LRU** chỉ quan tâm "lúc nào": `C` có `last = 4` nhỏ nhất → loại `C`.
- **LFU** quan tâm "bao nhiêu lần" trước: `C` có `cnt = 1` nhỏ nhất → cũng loại `C`. Ở ví dụ này hai chính sách trùng nhau, nhưng ở mục 3 ta sẽ thấy một ví dụ chúng chọn **khác** nhau.
- `min(..., key=...)` sẽ được giải thích kỹ ở mục 3. Tạm hiểu: "tìm phần tử có giá trị `key(...)` nhỏ nhất".

### 2.2 `dict` — tra cứu `O(1)` trung bình và **giữ thứ tự chèn**

`dict` (dictionary, bảng băm) là "cuốn sổ" ghi cặp `key → value`:

- `d[k] = v` — ghi / ghi đè; `d[k]` — đọc; `k in d` — kiểm tra có không; `del d[k]` hay `d.pop(k)` — xóa. Tất cả đều `O(1)` **trung bình** (hiếm khi va chạm băm thì chậm hơn, nên ta luôn nói "trung bình").
- Từ Python 3.7, `dict` **nhớ thứ tự chèn**: duyệt `for k in d` sẽ ra key theo thứ tự được chèn lần đầu. Ghi đè `d[k] = v2` **không** đổi vị trí của `k`. Muốn đưa `k` về cuối thì phải `pop` rồi chèn lại.
- `next(iter(d))` lấy **key được chèn sớm nhất** (key đầu tiên) mà không cần duyệt hết.

In [3]:
d = {}
d["a"] = 1
d["b"] = 2
d["c"] = 3
print("Ban đầu          :", d)

d["a"] = 100                        # ghi đè: "a" vẫn đứng đầu
print("Ghi đè a         :", d)

v = d.pop("a")                      # xóa "a" và lấy lại giá trị
d["a"] = v                          # chèn lại -> "a" xuống cuối
print("pop rồi chèn lại :", d)

print("Key sớm nhất     :", next(iter(d)))
print("'b' in d ?       :", "b" in d, "| 'z' in d ?", "z" in d)
print("d.get('z', -1)   :", d.get("z", -1))

Ban đầu          : {'a': 1, 'b': 2, 'c': 3}
Ghi đè a         : {'a': 100, 'b': 2, 'c': 3}
pop rồi chèn lại : {'b': 2, 'c': 3, 'a': 100}
Key sớm nhất     : b
'b' in d ?       : True | 'z' in d ? False
d.get('z', -1)   : -1


### 2.3 Class, `self` và "object nhớ trạng thái"

Ở bài thường, một hàm nhận input rồi trả output, xong là quên. Ở bài **thiết kế class**, ta tạo **một object** rồi gọi method của nó **nhiều lần**; giữa các lần gọi, object phải **nhớ** dữ liệu cũ. Chỗ để nhớ là các **thuộc tính** (attribute) gắn với `self`:

- `__init__(self, ...)` chạy **một lần** lúc tạo object — nơi khởi tạo trạng thái (vd `self.data = {}`).
- Mỗi method nhận `self` là chính object đó, nên đọc/ghi được `self.data`.

Ví dụ nhỏ: một bộ đếm lượt bấm cho từng nút.

In [4]:
class ClickCounter:
    def __init__(self, start: int):
        self.start = start
        self.counts = {}             # trạng thái được nhớ giữa các lần gọi

    def click(self, button: str) -> None:
        self.counts[button] = self.counts.get(button, self.start) + 1

    def total(self, button: str) -> int:
        return self.counts.get(button, self.start)


cc = ClickCounter(0)
cc.click("ok")
cc.click("ok")
cc.click("cancel")
print(cc.total("ok"), cc.total("cancel"), cc.total("help"))   # 2 1 0
print(cc.counts)

2 1 0
{'ok': 2, 'cancel': 1}


### 2.4 Chạy chuỗi lệnh kiểu LeetCode: `getattr` và hàm `run_commands`

LeetCode nhận 2 list `commands`, `args` rồi tự gọi method theo **tên dạng chuỗi**. Trong Python, `getattr(obj, "put")` trả về method `obj.put` khi chỉ biết tên là chuỗi `"put"`. Còn `f(*a)` "rải" list `a = [1, 1]` thành `f(1, 1)`.

Ta viết `run_commands(cls, commands, args)`:

1. `obj = cls(*args[0])` — lệnh đầu là constructor (ta truyền **class** vào, nên chạy được với mọi phiên bản `LFUCacheScan`, `LFUCacheHeap`, … dù tên lệnh đầu là `"LFUCache"`).
2. Với mỗi lệnh còn lại: `getattr(obj, name)(*a)`, gom kết quả vào list (hàm `void` trả `None`, khớp với `null`).

In [5]:
f = getattr(cc, "click")      # lấy method theo tên chuỗi
f("ok")                       # tương đương cc.click("ok")
a = ["ok"]
print(getattr(cc, "total")(*a))   # *a rải list thành tham số -> cc.total("ok") = 3


def run_commands(cls, commands, args):
    # Chạy chuỗi lệnh kiểu LeetCode, trả về list output (None cho hàm void).
    obj = cls(*args[0])
    out = [None]
    for name, a in zip(commands[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out


def fmt_call(name, a):
    # "put", [1, 1] -> "put(1, 1)" (để in trace cho gọn)
    return f"{name}({', '.join(map(str, a))})"


print(run_commands(ClickCounter, ["ClickCounter", "click", "click", "total", "total"],
                   [[10], ["x"], ["x"], ["x"], ["y"]]))     # [None, None, None, 12, 10]

3
[None, None, None, 12, 10]


### 2.5 `collections.OrderedDict` — dict có thứ tự **điều khiển được**

`OrderedDict` giống `dict` nhưng có thêm 2 thao tác `O(1)` cực hợp để làm cache:

| Thao tác | Ý nghĩa | Độ phức tạp |
|---|---|---|
| `od.move_to_end(k)` | đưa key `k` về **cuối** (vị trí "mới nhất") | `O(1)` |
| `od.move_to_end(k, last=False)` | đưa `k` về **đầu** | `O(1)` |
| `od.popitem(last=False)` | lấy ra và xóa cặp **đầu tiên** (cũ nhất) | `O(1)` |
| `od.popitem()` (mặc định `last=True`) | lấy ra và xóa cặp **cuối** (mới nhất) | `O(1)` |

Quy ước suốt notebook: **bên trái = cũ nhất, bên phải = mới nhất**. Như vậy một `OrderedDict` chính là một "hàng đợi theo độ mới": mỗi lần dùng key thì đẩy nó về phải, khi cần loại thì lấy từ trái.

Vì sao không dùng `list`? Vì `list.remove(k)` phải **dò** từ đầu để tìm `k` → `O(n)`. `OrderedDict` vừa tra key `O(1)` (nó là dict), vừa giữ thứ tự (bên trong có một danh sách liên kết đôi — ta sẽ tự cài ở Cách 4).

In [6]:
from collections import OrderedDict

od = OrderedDict()
od[1] = "một"
od[2] = "hai"
od[3] = "ba"
print("Ban đầu           :", list(od))

od.move_to_end(1)                    # "dùng" key 1 -> 1 thành mới nhất
print("move_to_end(1)    :", list(od))

oldest = od.popitem(last=False)      # lấy cặp cũ nhất (bên trái)
print("popitem(last=False):", oldest, "-> còn", list(od))

od[4] = "bốn"                        # key mới chèn vào bên phải
print("chèn 4            :", list(od))
print("rỗng chưa?        :", not od, "| len =", len(od))

Ban đầu           : [1, 2, 3]
move_to_end(1)    : [2, 3, 1]
popitem(last=False): (2, 'hai') -> còn [3, 1]
chèn 4            : [3, 1, 4]
rỗng chưa?        : False | len = 3


### 2.6 `collections.defaultdict` — dict tự tạo giá trị mặc định

Ở cách chính ta cần một dict `freq → OrderedDict`. Mỗi lần một key lên tần suất mới `f`, nếu chưa có "tầng" `f` thì phải tạo `OrderedDict()` rỗng trước. `defaultdict(OrderedDict)` làm việc đó tự động: truy cập `dd[f]` mà `f` chưa có → tự tạo `OrderedDict()` rỗng.

**Bẫy cần nhớ:** chỉ cần **đọc** `dd[f]` cũng đã tạo ra một mục rỗng. Vì vậy khi chỉ muốn **kiểm tra** thì dùng `f in dd`, đừng dùng `dd[f]`.

In [7]:
from collections import defaultdict

dd = defaultdict(OrderedDict)
dd[1]["a"] = None          # tầng 1 chưa có -> tự tạo OrderedDict rồi thêm "a"
dd[1]["b"] = None
dd[2]["c"] = None
print({f: list(b) for f, b in dd.items()})

print(3 in dd)             # False: kiểm tra bằng `in` không tạo gì
_ = dd[3]                  # chỉ ĐỌC thôi cũng tạo tầng 3 rỗng!
print(3 in dd, {f: list(b) for f, b in dd.items()})
del dd[3]                  # dọn tầng rỗng

{1: ['a', 'b'], 2: ['c']}
False
True {1: ['a', 'b'], 2: ['c'], 3: []}


### 2.7 Ôn lại LRU Cache (LeetCode #146) bằng `OrderedDict`

Bài 146 là "em ruột" của bài này: cùng `get` / `put`, cùng `capacity`, nhưng khi đầy thì loại key **lâu nhất chưa dùng** (không quan tâm số lần dùng).

Ví dụ của #146:

```text
["LRUCache","put","put","get","put","get","put","get","get","get"]
[[2],[1,1],[2,2],[1],[3,3],[2],[4,4],[1],[3],[4]]
Output: [null,null,null,1,null,-1,null,-1,3,4]
```

Ý tưởng: một `OrderedDict` `key → value`, **trái = lâu chưa dùng nhất, phải = vừa dùng**.

- `get(k)`: có thì `move_to_end(k)` (vừa dùng) rồi trả value; không có trả `-1`.
- `put(k, v)`: có rồi thì ghi đè + `move_to_end(k)`; chưa có thì nếu đầy `popitem(last=False)` (loại bên trái), rồi chèn `k` vào bên phải.

| Bước | Lệnh | Trả về | Thứ tự (trái = cũ nhất) | Ghi chú |
|---:|---|---|---|---|
| 1 | `put(1,1)` | null | `[1]` | |
| 2 | `put(2,2)` | null | `[1, 2]` | |
| 3 | `get(1)` | 1 | `[2, 1]` | 1 vừa dùng → sang phải |
| 4 | `put(3,3)` | null | `[1, 3]` | đầy → loại trái nhất là 2 |
| 5 | `get(2)` | -1 | `[1, 3]` | |
| 6 | `put(4,4)` | null | `[3, 4]` | đầy → loại 1 |
| 7 | `get(1)` | -1 | `[3, 4]` | |
| 8 | `get(3)` | 3 | `[4, 3]` | |
| 9 | `get(4)` | 4 | `[3, 4]` | |

In [8]:
class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.od = OrderedDict()          # key -> value, trái = lâu chưa dùng nhất

    def get(self, key: int) -> int:
        if key not in self.od:
            return -1
        self.od.move_to_end(key)         # vừa dùng -> về bên phải
        return self.od[key]

    def put(self, key: int, value: int) -> None:
        if key in self.od:
            self.od[key] = value
            self.od.move_to_end(key)
            return
        if len(self.od) == self.capacity:
            self.od.popitem(last=False)  # loại key lâu chưa dùng nhất
        self.od[key] = value


lru_cmds = ["LRUCache", "put", "put", "get", "put", "get", "put", "get", "get", "get"]
lru_args = [[2], [1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [4]]
assert run_commands(LRUCache, lru_cmds, lru_args) == [None, None, None, 1, None, -1, None, -1, 3, 4]

# Trace: in thứ tự key sau mỗi lệnh
lru = LRUCache(2)
for step, (name, a) in enumerate(zip(lru_cmds[1:], lru_args[1:]), start=1):
    res = getattr(lru, name)(*a)
    print(f"{step}. {fmt_call(name, a):<9} -> {str(res):<4} thứ tự = {list(lru.od)}")

1. put(1, 1) -> None thứ tự = [1]
2. put(2, 2) -> None thứ tự = [1, 2]
3. get(1)    -> 1    thứ tự = [2, 1]
4. put(3, 3) -> None thứ tự = [1, 3]
5. get(2)    -> -1   thứ tự = [1, 3]
6. put(4, 4) -> None thứ tự = [3, 4]
7. get(1)    -> -1   thứ tự = [3, 4]
8. get(3)    -> 3    thứ tự = [4, 3]
9. get(4)    -> 4    thứ tự = [3, 4]


Output khớp đúng bảng ở trên. Ghi nhớ 2 "động tác" của LRU bằng `OrderedDict`: **dùng → đẩy về phải**, **loại → lấy bên trái**. Ở cách chính của LFU, ta sẽ dùng lại đúng 2 động tác này, nhưng **trên từng tầng tần suất** một.

**LRU khác LFU ở đâu?** LRU chỉ nhìn "lần cuối dùng là khi nào". LFU nhìn "đã dùng **bao nhiêu lần**" trước, chỉ khi hòa mới nhìn "khi nào". Một key được dùng rất nhiều lần nhưng **gần đây** không dùng: LRU sẵn sàng loại nó, LFU thì giữ lại. Ta sẽ chạy một ví dụ minh họa ngay sau khi có Cách 1.

## 3. Cách 1 — `dict` + quét tìm key ít dùng nhất (dễ hiểu, làm trọng tài)

Đây là cách một người mới tự nghĩ ra: **ghi chép đầy đủ, khi cần thì tìm**.

- Một dict `data`: `key → [value, cnt, last]`.
- Một đồng hồ logic `clock`: mỗi lần một key được **dùng** (put key mới, put key cũ, get trúng) thì `clock += 1` và gán `last = clock`.
- `get(k)`: không có → `-1`. Có → tăng `cnt`, cập nhật `last`, trả value.
- `put(k, v)`: có rồi → ghi value, tăng `cnt`, cập nhật `last`. Chưa có → nếu đầy thì **quét toàn bộ** để tìm key có `(cnt, last)` nhỏ nhất và xóa nó; rồi chèn `k` với `cnt = 1`.

### So sánh tuple — một dòng code xử lý luôn cả luật "hòa thì LRU"

Python so sánh hai tuple **từ trái sang phải**: so phần tử đầu; nếu bằng nhau mới so phần tử thứ hai, … Vì vậy `(cnt, last)` nhỏ nhất chính là: `cnt` nhỏ nhất, và nếu hòa `cnt` thì `last` nhỏ nhất (lâu chưa dùng nhất). Đúng y luật của đề!

`min(iterable, key=f)` trả về phần tử `x` có `f(x)` nhỏ nhất. `lambda k: ...` là cách viết một hàm ngắn không tên ngay tại chỗ.

In [9]:
print((1, 5) < (2, 1))     # True : so cnt trước, 1 < 2
print((2, 3) < (2, 5))     # True : cnt hòa (2 == 2) -> so last, 3 < 5
print((2, 5) < (2, 3))     # False

info = {1: (2, 3), 3: (2, 5)}               # key -> (cnt, last)   (giống bước put(4,4) của đề)
print(min(info, key=lambda k: info[k]))     # 1: cùng cnt=2, nhưng key 1 dùng lần cuối lúc 3 < 5

True
True
False
1


In [10]:
class LFUCacheScan:
    # Cách 1: dict + quét. Dễ hiểu, dùng làm "trọng tài" khi test chéo.

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data = {}          # key -> [value, cnt, last]
        self.clock = 0          # đồng hồ logic

    def _use(self, key: int) -> None:
        self.clock += 1
        item = self.data[key]
        item[1] += 1            # cnt tăng 1
        item[2] = self.clock    # dùng gần nhất = bây giờ

    def get(self, key: int) -> int:
        if key not in self.data:
            return -1
        self._use(key)
        return self.data[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.data:                # key cũ: cập nhật value + tính là 1 lần dùng
            self.data[key][0] = value
            self._use(key)
            return
        if len(self.data) == self.capacity: # đầy: quét tìm (cnt, last) nhỏ nhất
            victim = min(self.data, key=lambda k: (self.data[k][1], self.data[k][2]))
            del self.data[victim]
        self.clock += 1
        self.data[key] = [value, 1, self.clock]


out = run_commands(LFUCacheScan, commands_ex1, args_ex1)
print(out)
assert out == expected_ex1

[None, None, None, 1, None, -1, 3, None, -1, 3, 4]


### Trace Cách 1 trên Example 1

Mỗi ô `k:[v, cnt, last]`. Cột "Ứng viên" chỉ hiện khi phải loại: ta so các cặp `(cnt, last)`.

| Bước | Lệnh | `clock` sau | Trả về | Ứng viên `(cnt, last)` | `data` sau lệnh |
|---:|---|---:|---|---|---|
| 1 | `put(1,1)` | 1 | null | | `1:[1,1,1]` |
| 2 | `put(2,2)` | 2 | null | | `1:[1,1,1]  2:[2,1,2]` |
| 3 | `get(1)` | 3 | 1 | | `1:[1,2,3]  2:[2,1,2]` |
| 4 | `put(3,3)` | 4 | null | 1:(2,3), **2:(1,2)** → loại 2 | `1:[1,2,3]  3:[3,1,4]` |
| 5 | `get(2)` | 4 | -1 | | không đổi (trượt không tăng `clock`) |
| 6 | `get(3)` | 5 | 3 | | `1:[1,2,3]  3:[3,2,5]` |
| 7 | `put(4,4)` | 6 | null | **1:(2,3)**, 3:(2,5) → hòa cnt, loại 1 | `3:[3,2,5]  4:[4,1,6]` |
| 8 | `get(1)` | 6 | -1 | | không đổi |
| 9 | `get(3)` | 7 | 3 | | `3:[3,3,7]  4:[4,1,6]` |
| 10 | `get(4)` | 8 | 4 | | `3:[3,3,7]  4:[4,2,8]` |

Cột `cnt` khớp đúng phần giải thích của đề (`cnt(3)=3`, `cnt(4)=2` ở cuối). Cell dưới in lại bảng này bằng code.

In [11]:
def trace_scan(commands, args):
    cache = LFUCacheScan(*args[0])
    for step, (name, a) in enumerate(zip(commands[1:], args[1:]), start=1):
        cand = ""
        if name == "put" and a[0] not in cache.data and len(cache.data) == cache.capacity:
            cand = "ứng viên " + " ".join(f"{k}:{(v[1], v[2])}" for k, v in cache.data.items())
        res = getattr(cache, name)(*a)
        state = "  ".join(f"{k}:{v}" for k, v in cache.data.items())
        print(f"{step:>2}. {fmt_call(name, a):<9} clock={cache.clock} -> {str(res):<4} | {state:<22} {cand}")


trace_scan(commands_ex1, args_ex1)

 1. put(1, 1) clock=1 -> None | 1:[1, 1, 1]            
 2. put(2, 2) clock=2 -> None | 1:[1, 1, 1]  2:[2, 1, 2] 
 3. get(1)    clock=3 -> 1    | 1:[1, 2, 3]  2:[2, 1, 2] 
 4. put(3, 3) clock=4 -> None | 1:[1, 2, 3]  3:[3, 1, 4] ứng viên 1:(2, 3) 2:(1, 2)
 5. get(2)    clock=4 -> -1   | 1:[1, 2, 3]  3:[3, 1, 4] 
 6. get(3)    clock=5 -> 3    | 1:[1, 2, 3]  3:[3, 2, 5] 
 7. put(4, 4) clock=6 -> None | 3:[3, 2, 5]  4:[4, 1, 6] ứng viên 1:(2, 3) 3:(2, 5)
 8. get(1)    clock=6 -> -1   | 3:[3, 2, 5]  4:[4, 1, 6] 
 9. get(3)    clock=7 -> 3    | 3:[3, 3, 7]  4:[4, 1, 6] 
10. get(4)    clock=8 -> 4    | 3:[3, 3, 7]  4:[4, 2, 8] 


### LRU và LFU chọn khác nhau — một ví dụ

`capacity = 2`: `put(1,1)`, `get(1)`, `get(1)`, `put(2,2)`, `put(3,3)`, rồi hỏi `get(1)`, `get(2)`.

- Đến lúc `put(3,3)`: key 1 đã dùng **3 lần** nhưng lần cuối lúc 3; key 2 dùng **1 lần**, lần cuối lúc 4.
- **LRU** loại key lâu chưa dùng nhất → loại **1** → sau đó `get(1) = -1`, `get(2) = 2`.
- **LFU** loại key ít dùng nhất → loại **2** → sau đó `get(1) = 1`, `get(2) = -1`.

In [12]:
cmp_cmds = ["X", "put", "get", "get", "put", "put", "get", "get"]
cmp_args = [[2], [1, 1], [1], [1], [2, 2], [3, 3], [1], [2]]
print("LRU:", run_commands(LRUCache, cmp_cmds, cmp_args))
print("LFU:", run_commands(LFUCacheScan, cmp_cmds, cmp_args))

LRU: [None, None, 1, 1, None, None, -1, 2]
LFU: [None, None, 1, 1, None, None, 1, -1]


### Độ phức tạp Cách 1 và chỗ vướng

- `get`: tra dict + cập nhật 2 số → `O(1)` trung bình.
- `put` key cũ: `O(1)`.
- `put` key mới khi **đầy**: `min(...)` duyệt **toàn bộ** `c` key → `O(c)`.
- Bộ nhớ: `O(c)`.

Trường hợp xấu nhất: cache luôn đầy và gần như mọi lệnh là `put` key mới. Khi đó tổng công việc khoảng `q · c = 2·10^5 · 10^4 = 2·10^9` phép so sánh — quá chậm (Python chỉ làm cỡ `10^7` phép đơn giản mỗi giây). Đề lại đòi `O(1)`.

**Nút thắt:** mỗi lần cần loại ta **tìm lại từ đầu** "key nhỏ nhất theo `(cnt, last)`", dù giữa hai lần loại chỉ có vài key thay đổi. Ta cần một cấu trúc **luôn biết sẵn** phần tử nhỏ nhất. Cấu trúc kinh điển cho việc đó là **heap**.

Dù chậm, Cách 1 **rất khó viết sai**, nên ta giữ nó làm **trọng tài**: mọi cách nhanh hơn phía sau đều phải cho output y hệt Cách 1 trên hàng trăm chuỗi lệnh ngẫu nhiên.

## 4. Cách 2 — Heap `(freq, time, key)` + lazy deletion

### 4.1 Heap là gì? (`heapq` từ con số 0)

**Min-heap** là một cây nhị phân gần đầy, lưu trong một `list`, thỏa: **mỗi nút ≤ hai nút con**. Hệ quả quan trọng duy nhất ta cần: **phần tử nhỏ nhất luôn ở `heap[0]`**.

Module `heapq` của Python thao tác trên list thường:

| Lệnh | Ý nghĩa | Độ phức tạp |
|---|---|---|
| `heapq.heappush(h, x)` | thêm `x` | `O(log n)` |
| `heapq.heappop(h)` | lấy ra và xóa phần tử **nhỏ nhất** | `O(log n)` |
| `h[0]` | xem phần tử nhỏ nhất (không xóa) | `O(1)` |
| `heapq.heapify(h)` | biến list bất kỳ thành heap | `O(n)` |

Vì sao `O(log n)`? Cây gần đầy có chiều cao khoảng `log2(n)`; mỗi lần thêm/xóa, phần tử chỉ "nổi lên" hoặc "chìm xuống" dọc **một đường** từ gốc tới lá. Với `n = 10^4`, `log2(n) ≈ 14` bước.

Heap so sánh được cả **tuple** (theo luật so tuple ở mục 3), nên ta có thể đẩy vào `(freq, time, key)`: phần tử nhỏ nhất chính là key cần loại.

In [13]:
import heapq

h = []
for x in [5, 1, 8, 3, 2]:
    heapq.heappush(h, x)
print("list bên trong heap:", h, "| nhỏ nhất h[0] =", h[0])
print("pop lần lượt       :", [heapq.heappop(h) for _ in range(5)])

h = []
for entry in [(2, 3, 1), (1, 2, 2), (2, 5, 3)]:      # (freq, time, key)
    heapq.heappush(h, entry)
print("tuple nhỏ nhất     :", h[0], "-> key cần loại là", h[0][2])

list bên trong heap: [1, 2, 8, 5, 3] | nhỏ nhất h[0] = 1
pop lần lượt       : [1, 2, 3, 5, 8]
tuple nhỏ nhất     : (1, 2, 2) -> key cần loại là 2


### 4.2 Vấn đề: heap **không sửa được** phần tử ở giữa → lazy deletion

Mỗi lần key `k` được dùng, `freq` và `time` của nó thay đổi. Nhưng `heapq` không có thao tác "tìm bản ghi cũ của `k` rồi sửa" (tìm đã mất `O(n)`). Mẹo **lazy deletion** (xóa lười):

1. **Không sửa, không xóa** bản ghi cũ. Mỗi lần dùng `k`, cứ **đẩy thêm** một bản ghi mới `(freq_mới, time_mới, k)`.
2. Song song giữ một dict `info[k] = [value, freq, time]` là **sự thật hiện tại**.
3. Khi cần loại: `heappop` liên tục. Bản ghi `(f, t, k)` nào **không khớp** `info[k]` (key đã bị loại, hoặc `freq/time` đã đổi) là **bản ghi cũ (stale)** → vứt đi, pop tiếp. Bản ghi đầu tiên **khớp** chính là key cần loại.

Vì `time` tăng mãi, mỗi lần dùng tạo một `time` mới duy nhất → bản ghi khớp `info` của mỗi key là **duy nhất**. Cell dưới minh họa: key `"a"` được dùng 2 lần nên có 1 bản ghi cũ nằm trong heap.

In [14]:
info = {"a": [None, 2, 3], "b": [None, 1, 2]}        # key -> [value, freq, time] (sự thật hiện tại)
h = [(1, 1, "a"), (1, 2, "b"), (2, 3, "a")]          # (1,1,'a') là bản ghi cũ của "a"
heapq.heapify(h)

while True:
    f, t, k = heapq.heappop(h)
    ok = info[k][1] == f and info[k][2] == t
    print(f"pop {(f, t, k)} so với info[{k!r}]=(freq={info[k][1]}, time={info[k][2]}) ->",
          "KHỚP, loại key này" if ok else "cũ, bỏ qua")
    if ok:
        break

pop (1, 1, 'a') so với info['a']=(freq=2, time=3) -> cũ, bỏ qua
pop (1, 2, 'b') so với info['b']=(freq=1, time=2) -> KHỚP, loại key này


In [15]:
class LFUCacheHeap:
    # Cách 2: heap (freq, time, key) + lazy deletion + dọn rác định kỳ.

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.info = {}          # key -> [value, freq, time]  (sự thật hiện tại)
        self.heap = []          # (freq, time, key), có thể lẫn bản ghi cũ
        self.clock = 0

    def _push(self, key: int) -> None:
        _, f, t = self.info[key]
        heapq.heappush(self.heap, (f, t, key))
        if len(self.heap) > 2 * self.capacity + 8:      # quá nhiều rác -> xây lại heap
            self.heap = [(f, t, k) for k, (_, f, t) in self.info.items()]
            heapq.heapify(self.heap)

    def _is_valid(self, entry) -> bool:
        f, t, k = entry
        return k in self.info and self.info[k][1] == f and self.info[k][2] == t

    def _use(self, key: int) -> None:
        self.clock += 1
        item = self.info[key]
        item[1] += 1
        item[2] = self.clock
        self._push(key)

    def _evict(self) -> int:
        while True:
            entry = heapq.heappop(self.heap)
            if self._is_valid(entry):
                del self.info[entry[2]]
                return entry[2]
            # bản ghi cũ: bỏ qua

    def get(self, key: int) -> int:
        if key not in self.info:
            return -1
        self._use(key)
        return self.info[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.info:
            self.info[key][0] = value
            self._use(key)
            return
        if len(self.info) == self.capacity:
            self._evict()
        self.clock += 1
        self.info[key] = [value, 1, self.clock]
        self._push(key)


out = run_commands(LFUCacheHeap, commands_ex1, args_ex1)
print(out)
assert out == expected_ex1

[None, None, None, 1, None, -1, 3, None, -1, 3, 4]


### Trace Cách 2: chuyện gì xảy ra ở 2 lần loại của Example 1?

`time` ở đây trùng với `clock` của Cách 1. Bản ghi được đẩy vào theo thứ tự: `(1,1,1)` put 1, `(1,2,2)` put 2, `(2,3,1)` get 1, `(1,4,3)` put 3, `(2,5,3)` get 3.

| Lúc | Heap (sort cho dễ nhìn) | Pop | So với `info` | Kết luận |
|---|---|---|---|---|
| `put(3,3)` | `(1,1,1) (1,2,2) (2,3,1)` | `(1,1,1)` | `info[1]` có freq 2, time 3 | cũ → bỏ |
| | | `(1,2,2)` | `info[2]` có freq 1, time 2 | **khớp → loại key 2** |
| `put(4,4)` | `(1,4,3) (2,3,1) (2,5,3)` | `(1,4,3)` | `info[3]` có freq 2, time 5 | cũ → bỏ |
| | | `(2,3,1)` | `info[1]` có freq 2, time 3 | **khớp → loại key 1** |

Để ý lần thứ hai: bản ghi `(1,4,3)` có `freq = 1` nhỏ nhất heap nhưng là **rác** (key 3 đã lên freq 2). Nếu quên kiểm tra, ta sẽ loại nhầm key 3!

In [16]:
class LFUCacheHeapTrace(LFUCacheHeap):
    def _evict(self) -> int:
        print(f"    cần loại; heap = {sorted(self.heap)}")
        while True:
            entry = heapq.heappop(self.heap)
            k = entry[2]
            if self._is_valid(entry):
                print(f"    pop {entry}: khớp info[{k}] -> loại key {k}")
                del self.info[k]
                return k
            print(f"    pop {entry}: info[{k}] = (freq, time) {tuple(self.info[k][1:]) if k in self.info else 'đã bị loại'} -> cũ, bỏ")


cache = LFUCacheHeapTrace(2)
for name, a in zip(commands_ex1[1:], args_ex1[1:]):
    res = getattr(cache, name)(*a)
    print(f"{fmt_call(name, a)} -> {res}")

put(1, 1) -> None
put(2, 2) -> None
get(1) -> 1
    cần loại; heap = [(1, 1, 1), (1, 2, 2), (2, 3, 1)]
    pop (1, 1, 1): info[1] = (freq, time) (2, 3) -> cũ, bỏ
    pop (1, 2, 2): khớp info[2] -> loại key 2
put(3, 3) -> None
get(2) -> -1
get(3) -> 3
    cần loại; heap = [(1, 4, 3), (2, 3, 1), (2, 5, 3)]
    pop (1, 4, 3): info[3] = (freq, time) (2, 5) -> cũ, bỏ
    pop (2, 3, 1): khớp info[1] -> loại key 1
put(4, 4) -> None
get(1) -> -1
get(3) -> 3
get(4) -> 4


### Độ phức tạp Cách 2 và chỗ vướng

- Mỗi lần dùng key: 1 `heappush` → `O(log H)` với `H` = kích thước heap.
- Mỗi lần loại: vài `heappop`. Mỗi bản ghi chỉ bị pop **tối đa một lần** trong đời, nên tổng số pop ≤ tổng số push ≤ `q` → **amortized** (trung bình chia đều) `O(log H)` mỗi thao tác.
- Không dọn rác thì `H` có thể lên tới `q = 2·10^5` bản ghi. Ta dọn: khi `H > 2c + 8` thì xây lại heap chỉ từ các bản ghi hợp lệ (`heapify` `O(c)`); việc này xảy ra sau ít nhất khoảng `c` lần push nên chia đều chỉ thêm `O(1)` mỗi thao tác. Nhờ vậy `H = O(c)`, thời gian `O(log c)` amortized, bộ nhớ `O(c)`.

Cách 2 đủ nhanh để Accepted trên thực tế, nhưng **chưa phải `O(1)`** như đề đòi, và logic lazy deletion khá rối. Câu hỏi: heap là công cụ tổng quát cho "giá trị tùy ý thay đổi tùy ý". Bài này có gì **đặc biệt** để làm tốt hơn?

## 5. Bước đột phá: `freq` chỉ **tăng đúng 1** mỗi lần

Nhìn kỹ cách `freq` thay đổi, ta thấy 3 quan sát "vàng":

1. **Mỗi lần dùng, `freq` của key tăng đúng 1**: từ `f` lên `f + 1`. Không bao giờ nhảy cóc, không bao giờ giảm.
2. **Key mới luôn có `freq = 1`** — nhỏ nhất có thể. Nên ngay sau khi chèn key mới, tần suất nhỏ nhất của cả cache chắc chắn là `1`.
3. **Trong cùng một mức `freq`, ta cần thứ tự LRU** (luật hòa). Mà "LRU trong một nhóm" thì ta đã làm ở #146: một `OrderedDict`, trái = cũ nhất.

Từ đó: hãy **nhóm các key theo `freq`** thành từng **tầng** (bucket). Tầng `f` là một `OrderedDict` chứa các key có `freq = f`, xếp theo thứ tự **vào tầng** (trái = vào sớm nhất). Khi một key được dùng, nó **rời tầng `f`, xếp vào cuối tầng `f + 1`**.

Key cần loại = **key trái nhất của tầng có `f` nhỏ nhất**. Chỉ còn phải biết "tầng nhỏ nhất là tầng nào" trong `O(1)` → giữ một biến `min_freq`. Nhờ quan sát 1 và 2, `min_freq` chỉ thay đổi theo **2 quy tắc**:

- **Quy tắc A** (dùng key): key rời tầng `f` → nếu tầng `f` trở nên **rỗng** và `f == min_freq` thì `min_freq = f + 1`. Tại sao chắc chắn là `f + 1` mà không phải số lớn hơn? Vì chính key vừa dùng **đang nằm ở tầng `f + 1`**, nên tầng đó không rỗng.
- **Quy tắc B** (chèn key mới): `min_freq = 1`.

Cell dưới "nhìn" trạng thái của Cách 1 theo kiểu tầng: gom key theo `cnt`, trong mỗi tầng sắp theo `last`.

In [17]:
def layers_from_scan(cache: LFUCacheScan) -> str:
    groups = {}
    for k, (v, c, last) in sorted(cache.data.items(), key=lambda kv: kv[1][2]):  # sort theo last
        groups.setdefault(c, []).append(k)        # setdefault: chưa có tầng c thì tạo list rỗng
    return "  ".join(f"f{c}:{groups[c]}" for c in sorted(groups))


cache = LFUCacheScan(3)
for name, a in [("put", (1, 1)), ("put", (2, 2)), ("put", (3, 3)), ("get", (2,)), ("get", (1,)), ("get", (2,))]:
    getattr(cache, name)(*a)
    print(f"{fmt_call(name, a):<9} -> tầng: {layers_from_scan(cache)}")

put(1, 1) -> tầng: f1:[1]
put(2, 2) -> tầng: f1:[1, 2]
put(3, 3) -> tầng: f1:[1, 2, 3]
get(2)    -> tầng: f1:[1, 3]  f2:[2]
get(1)    -> tầng: f1:[3]  f2:[2, 1]
get(2)    -> tầng: f1:[3]  f2:[1]  f3:[2]


Đọc output và để ý **chuyển động** của key:

- Sau 3 lệnh `put`: cả 3 key ở tầng 1, theo thứ tự vào `[1, 2, 3]`.
- `get(2)`: key 2 **rời** tầng 1, xếp vào **cuối** tầng 2 → `f1:[1, 3]  f2:[2]`.
- `get(1)`: key 1 rời tầng 1, xếp **sau** key 2 ở tầng 2 → `f2:[2, 1]`.
- `get(2)`: key 2 lên tầng 3. Tầng 2 còn `[1]`.

Mỗi lần dùng chỉ đụng tới **2 tầng kề nhau** (`f` và `f + 1`) và chỉ thao tác ở **cuối** tầng mới — toàn thao tác `O(1)` của `OrderedDict`. Không cần quét, không cần heap.

### Thiết kế cấu trúc dữ liệu của Cách 3

```text
key_to_vf     : dict          key  -> (value, freq)
freq_to_keys  : defaultdict   freq -> OrderedDict{key: None}   (trái = cũ nhất trong tầng)
min_freq      : int           freq nhỏ nhất đang có trong cache

Ví dụ (ngay sau bước put(3,3) của Example 1):
  key_to_vf    = {1: (1, 2), 3: (3, 1)}
  freq_to_keys :
      f=1 : [3]      <- min_freq = 1 : nếu phải loại, lấy key TRÁI NHẤT ở đây
      f=2 : [1]
```

- `OrderedDict` của mỗi tầng chỉ dùng như một **"tập hợp có thứ tự"**: value để `None`, ta chỉ quan tâm key và thứ tự.
- Ta **xóa hẳn** tầng nào trở nên rỗng, để `freq_to_keys` không chứa rác (và để in trace cho gọn).

### 5.1 Động tác 1 — `_touch(key)`: dịch key từ tầng `f` lên tầng `f + 1`

Đây là trái tim của thuật toán, dùng chung cho `get` trúng và `put` key cũ:

1. Đọc `(value, f) = key_to_vf[key]`.
2. Xóa `key` khỏi tầng `f` (`del bucket[key]` — `O(1)` vì tầng là dict).
3. Nếu tầng `f` rỗng: xóa tầng; **nếu `min_freq == f` thì `min_freq = f + 1`** (Quy tắc A).
4. Thêm `key` vào **cuối** tầng `f + 1` (`freq_to_keys[f + 1][key] = None`).
5. Ghi `key_to_vf[key] = (value, f + 1)`.

Ví dụ số: trạng thái `f1:[3]  f2:[1]`, `min_freq = 1`, gọi `_touch(3)`:
tầng 1 rỗng và `1 == min_freq` → `min_freq = 2`; key 3 vào cuối tầng 2 → `f2:[1, 3]`. Đây chính là bước `get(3)` của đề. Cell dưới chạy đúng các bước này trên dict "trần" để thấy rõ.

In [18]:
def touch(key, key_to_vf, freq_to_keys, min_freq):
    value, f = key_to_vf[key]
    bucket = freq_to_keys[f]
    del bucket[key]                         # bước 2
    if not bucket:                          # bước 3
        del freq_to_keys[f]
        if min_freq == f:
            min_freq = f + 1
    freq_to_keys[f + 1][key] = None         # bước 4: vào cuối tầng f+1
    key_to_vf[key] = (value, f + 1)         # bước 5
    return min_freq


def show(freq_to_keys):
    return "  ".join(f"f{f}:{list(b)}" for f, b in sorted(freq_to_keys.items()))


key_to_vf = {1: (1, 2), 3: (3, 1)}
freq_to_keys = defaultdict(OrderedDict)
freq_to_keys[1][3] = None
freq_to_keys[2][1] = None
min_freq = 1
print("trước     :", show(freq_to_keys), "| min_freq =", min_freq)
min_freq = touch(3, key_to_vf, freq_to_keys, min_freq)
print("touch(3)  :", show(freq_to_keys), "| min_freq =", min_freq, "|", key_to_vf)
min_freq = touch(1, key_to_vf, freq_to_keys, min_freq)
print("touch(1)  :", show(freq_to_keys), "| min_freq =", min_freq, "| tầng 2 còn [3] nên min giữ 2")

trước     : f1:[3]  f2:[1] | min_freq = 1
touch(3)  : f2:[1, 3] | min_freq = 2 | {1: (1, 2), 3: (3, 2)}
touch(1)  : f2:[3]  f3:[1] | min_freq = 2 | tầng 2 còn [3] nên min giữ 2


### 5.2 Động tác 2 — `get` và `put`

**`get(key)`**

- Không có trong `key_to_vf` → trả `-1` (trượt **không** tính là một lần dùng, không đổi gì).
- Có → `_touch(key)` rồi trả value.

**`put(key, value)`** — có 3 trường hợp, **thứ tự kiểm tra rất quan trọng**:

1. `capacity == 0` → không làm gì (phòng thủ cho bản đề cũ).
2. `key` **đã có** → ghi value mới rồi `_touch(key)`. **Không loại ai**, kể cả khi cache đang đầy (số key không đổi).
3. `key` **mới**:
   - Nếu đầy: loại `freq_to_keys[min_freq].popitem(last=False)` (trái nhất của tầng nhỏ nhất), xóa khỏi `key_to_vf`; tầng rỗng thì xóa tầng.
   - **Sau đó** mới chèn: `key_to_vf[key] = (value, 1)`, thêm vào cuối tầng 1, `min_freq = 1` (Quy tắc B).

Vì sao **loại trước, chèn sau**? Nếu chèn trước thì `min_freq` đã thành 1 và tầng 1 có thể **chỉ chứa đúng key vừa chèn** → ta loại luôn key mới! (Mục 12 có code sai minh họa.) Và khi loại, ta không cần cập nhật `min_freq` cho đúng, vì ngay sau đó Quy tắc B đặt nó về 1.

## 6. Cách 3 — Các tầng tần suất bằng `OrderedDict` + `min_freq` (cách chính, `O(1)`)

Ghép các động tác ở mục 5 lại thành class hoàn chỉnh.

In [19]:
class LFUCacheBuckets:
    # Cách 3: key -> (value, freq)  +  freq -> OrderedDict(key)  +  min_freq.

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.key_to_vf = {}                              # key -> (value, freq)
        self.freq_to_keys = defaultdict(OrderedDict)     # freq -> {key: None}, trái = cũ nhất
        self.min_freq = 0

    def _touch(self, key: int) -> None:
        value, f = self.key_to_vf[key]
        bucket = self.freq_to_keys[f]
        del bucket[key]
        if not bucket:                                   # tầng f rỗng
            del self.freq_to_keys[f]
            if self.min_freq == f:                       # Quy tắc A
                self.min_freq = f + 1
        self.freq_to_keys[f + 1][key] = None             # vào cuối tầng f+1
        self.key_to_vf[key] = (value, f + 1)

    def get(self, key: int) -> int:
        if key not in self.key_to_vf:
            return -1
        self._touch(key)
        return self.key_to_vf[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.key_to_vf:                        # key cũ: cập nhật + 1 lần dùng
            _, f = self.key_to_vf[key]
            self.key_to_vf[key] = (value, f)
            self._touch(key)
            return
        if len(self.key_to_vf) == self.capacity:         # đầy: loại TRƯỚC
            bucket = self.freq_to_keys[self.min_freq]
            victim, _ = bucket.popitem(last=False)       # trái nhất của tầng nhỏ nhất
            if not bucket:
                del self.freq_to_keys[self.min_freq]
            del self.key_to_vf[victim]
        self.key_to_vf[key] = (value, 1)                 # chèn SAU
        self.freq_to_keys[1][key] = None
        self.min_freq = 1                                # Quy tắc B


out = run_commands(LFUCacheBuckets, commands_ex1, args_ex1)
print(out)
assert out == expected_ex1

[None, None, None, 1, None, -1, 3, None, -1, 3, 4]


## 7. Trace Cách 3 — bảng trạng thái các tầng tần suất

### 7.1 Example 1 (`capacity = 2`)

| Bước | Lệnh | Trả về | `key_to_vf` (key: (value, freq)) | Các tầng (trái = cũ nhất) | `min_freq` | Chuyện gì xảy ra |
|---:|---|---|---|---|:---:|---|
| 1 | `put(1,1)` | null | `1:(1,1)` | `f1:[1]` | 1 | key mới, chưa đầy → vào tầng 1 |
| 2 | `put(2,2)` | null | `1:(1,1) 2:(2,1)` | `f1:[1,2]` | 1 | key mới → cuối tầng 1 |
| 3 | `get(1)` | 1 | `1:(1,2) 2:(2,1)` | `f1:[2] f2:[1]` | 1 | 1 lên tầng 2; tầng 1 còn `[2]` → min giữ 1 |
| 4 | `put(3,3)` | null | `1:(1,2) 3:(3,1)` | `f1:[3] f2:[1]` | 1 | đầy → loại trái nhất tầng 1 là **2**; chèn 3 vào tầng 1 |
| 5 | `get(2)` | -1 | (không đổi) | `f1:[3] f2:[1]` | 1 | trượt |
| 6 | `get(3)` | 3 | `1:(1,2) 3:(3,2)` | `f2:[1,3]` | **2** | tầng 1 rỗng và là min → Quy tắc A: min = 2 |
| 7 | `put(4,4)` | null | `3:(3,2) 4:(4,1)` | `f1:[4] f2:[3]` | 1 | đầy → tầng min=2 là `[1,3]`, loại trái nhất là **1** (hòa freq, 1 cũ hơn) |
| 8 | `get(1)` | -1 | (không đổi) | `f1:[4] f2:[3]` | 1 | trượt |
| 9 | `get(3)` | 3 | `3:(3,3) 4:(4,1)` | `f1:[4] f3:[3]` | 1 | tầng 2 rỗng nhưng `2 != min_freq` → min giữ 1 |
| 10 | `get(4)` | 4 | `3:(3,3) 4:(4,2)` | `f2:[4] f3:[3]` | **2** | tầng 1 rỗng và là min → min = 2 |

Bước 7 là chỗ đáng xem nhất: hai key 1 và 3 cùng `freq = 2`, cùng nằm ở tầng 2, nhưng 1 vào tầng 2 **trước** (bước 3) còn 3 vào **sau** (bước 6). Thứ tự trong tầng chính là thứ tự "dùng gần nhất", nên trái nhất = LRU. Hàm `trace_lfu` dưới in đúng bảng này (cột cuối ghi key bị loại nếu có).

In [20]:
def layers_str(cache) -> str:
    return "  ".join(f"f{f}:{list(cache.freq_to_keys[f])}" for f in sorted(cache.freq_to_keys)) or "(rỗng)"


def trace_lfu(commands, args):
    cache = LFUCacheBuckets(*args[0])
    print(f"LFUCache({args[0][0]})")
    for step, (name, a) in enumerate(zip(commands[1:], args[1:]), start=1):
        before = set(cache.key_to_vf)
        res = getattr(cache, name)(*a)
        evicted = before - set(cache.key_to_vf)
        vf = " ".join(f"{k}:{vf}" for k, vf in cache.key_to_vf.items())
        note = f"loại {evicted.pop()}" if evicted else ""
        print(f"{step:>2}. {fmt_call(name, a):<10} -> {str(res):<4} | {vf:<26} | {layers_str(cache):<22} | min={cache.min_freq} {note}")


trace_lfu(commands_ex1, args_ex1)

LFUCache(2)
 1. put(1, 1)  -> None | 1:(1, 1)                   | f1:[1]                 | min=1 
 2. put(2, 2)  -> None | 1:(1, 1) 2:(2, 1)          | f1:[1, 2]              | min=1 
 3. get(1)     -> 1    | 1:(1, 2) 2:(2, 1)          | f1:[2]  f2:[1]         | min=1 
 4. put(3, 3)  -> None | 1:(1, 2) 3:(3, 1)          | f1:[3]  f2:[1]         | min=1 loại 2
 5. get(2)     -> -1   | 1:(1, 2) 3:(3, 1)          | f1:[3]  f2:[1]         | min=1 
 6. get(3)     -> 3    | 1:(1, 2) 3:(3, 2)          | f2:[1, 3]              | min=2 
 7. put(4, 4)  -> None | 3:(3, 2) 4:(4, 1)          | f1:[4]  f2:[3]         | min=1 loại 1
 8. get(1)     -> -1   | 3:(3, 2) 4:(4, 1)          | f1:[4]  f2:[3]         | min=1 
 9. get(3)     -> 3    | 3:(3, 3) 4:(4, 1)          | f1:[4]  f3:[3]         | min=1 
10. get(4)     -> 4    | 3:(3, 3) 4:(4, 2)          | f2:[4]  f3:[3]         | min=2 


### 7.2 Trace thứ hai — các nhánh dễ sai

`capacity = 2`: `put(1,1)`, `put(2,2)`, `get(1)`, `get(2)`, `put(3,3)`, `put(3,30)`, `put(4,4)`, `get(2)`, `get(3)`, `get(4)`.

Input này cố tình chạm 3 nhánh mà Example 1 chưa có:

- **`min_freq` tăng khi tầng 1 rỗng mà không có key mới nào chen vào** (bước 4): sau `get(1)`, `get(2)` mọi key đều ở tầng 2.
- **Loại ở tầng `min_freq = 2`** chứ không phải tầng 1 (bước 5).
- **`put` key đã có khi cache đầy** (bước 6): chỉ cập nhật value và tăng `freq`, **không loại ai**.

| Bước | Lệnh | Trả về | Các tầng sau lệnh | `min_freq` | Giải thích |
|---:|---|---|---|:---:|---|
| 1 | `put(1,1)` | null | `f1:[1]` | 1 | |
| 2 | `put(2,2)` | null | `f1:[1,2]` | 1 | |
| 3 | `get(1)` | 1 | `f1:[2] f2:[1]` | 1 | |
| 4 | `get(2)` | 2 | `f2:[1,2]` | 2 | tầng 1 rỗng, là min → 2 |
| 5 | `put(3,3)` | null | `f1:[3] f2:[2]` | 1 | đầy → loại trái nhất tầng 2 là **1**; chèn 3 |
| 6 | `put(3,30)` | null | `f2:[2,3]` | 2 | key cũ: value 30, 3 lên tầng 2 (đứng sau 2); tầng 1 rỗng → min 2; **không loại** |
| 7 | `put(4,4)` | null | `f1:[4] f2:[3]` | 1 | đầy → loại trái nhất tầng 2 là **2** |
| 8 | `get(2)` | -1 | `f1:[4] f2:[3]` | 1 | đã bị loại |
| 9 | `get(3)` | 30 | `f1:[4] f3:[3]` | 1 | trả value **mới** 30 |
| 10 | `get(4)` | 4 | `f2:[4] f3:[3]` | 2 | |

In [21]:
commands_t2 = ["LFUCache", "put", "put", "get", "get", "put", "put", "put", "get", "get", "get"]
args_t2 = [[2], [1, 1], [2, 2], [1], [2], [3, 3], [3, 30], [4, 4], [2], [3], [4]]
expected_t2 = [None, None, None, 1, 2, None, None, None, -1, 30, 4]

trace_lfu(commands_t2, args_t2)
assert run_commands(LFUCacheBuckets, commands_t2, args_t2) == expected_t2
assert run_commands(LFUCacheScan, commands_t2, args_t2) == expected_t2     # trọng tài đồng ý

LFUCache(2)
 1. put(1, 1)  -> None | 1:(1, 1)                   | f1:[1]                 | min=1 
 2. put(2, 2)  -> None | 1:(1, 1) 2:(2, 1)          | f1:[1, 2]              | min=1 
 3. get(1)     -> 1    | 1:(1, 2) 2:(2, 1)          | f1:[2]  f2:[1]         | min=1 
 4. get(2)     -> 2    | 1:(1, 2) 2:(2, 2)          | f2:[1, 2]              | min=2 
 5. put(3, 3)  -> None | 2:(2, 2) 3:(3, 1)          | f1:[3]  f2:[2]         | min=1 loại 1
 6. put(3, 30) -> None | 2:(2, 2) 3:(30, 2)         | f2:[2, 3]              | min=2 
 7. put(4, 4)  -> None | 3:(30, 2) 4:(4, 1)         | f1:[4]  f2:[3]         | min=1 loại 2
 8. get(2)     -> -1   | 3:(30, 2) 4:(4, 1)         | f1:[4]  f2:[3]         | min=1 
 9. get(3)     -> 30   | 3:(30, 3) 4:(4, 1)         | f1:[4]  f3:[3]         | min=1 
10. get(4)     -> 4    | 3:(30, 3) 4:(4, 2)         | f2:[4]  f3:[3]         | min=2 


## 8. Cách 4 (tùy chọn) — Tự cài danh sách liên kết đôi thay cho `OrderedDict`

Trong phỏng vấn, đôi khi người phỏng vấn hỏi: "Nếu ngôn ngữ không có `OrderedDict` thì sao?" (ví dụ C++ / Java phải tự ghép `HashMap` + `LinkedList`). Bên trong `OrderedDict` thực chất là **dict + danh sách liên kết đôi**. Ta tự cài để hiểu vì sao mọi thao tác đều `O(1)`.

### 8.1 Danh sách liên kết đôi (doubly linked list) là gì?

- Mỗi **nút** (node) giữ dữ liệu và **hai con trỏ**: `prev` (nút trước) và `next` (nút sau).
- Dùng 2 nút **canh gác** (sentinel) `head` và `tail` không chứa dữ liệu, luôn ở hai đầu: `head ⇄ A ⇄ B ⇄ tail`. Nhờ có canh gác, khi thêm/xóa ta **không bao giờ** phải kiểm tra "danh sách rỗng không? nút này có phải nút đầu không?".
- **Xóa một nút khi đã cầm nút trong tay**: nối `node.prev` với `node.next` → `O(1)`, không cần dò. Đây là điều `list` của Python không làm được (xóa giữa list phải dịch các phần tử sau → `O(n)`).
- **Thêm vào cuối**: chèn ngay trước `tail` → `O(1)`. **Lấy nút cũ nhất**: `head.next` → `O(1)`.

Để "cầm nút trong tay" ngay lập tức, ta giữ dict `key → Node`. Vẫn quy ước: **gần `head` = cũ nhất, gần `tail` = mới nhất**.

```text
head ⇄ [k=1] ⇄ [k=3] ⇄ tail          remove([k=1]):   head ⇄ [k=3] ⇄ tail
         ^ cũ nhất    ^ mới nhất                        (head.next = [k=3]; [k=3].prev = head)
```

### 8.2 LFU bằng danh sách liên kết đôi

Thay `freq_to_keys: freq → OrderedDict` bằng `freq_lists: freq → DLinkedList`, và `key_to_vf` bằng `nodes: key → Node` (mỗi nút tự giữ `value`, `freq`). Logic **y hệt** Cách 3:

- `_touch(node)`: gỡ nút khỏi list của `node.freq`; list rỗng → xóa, và nếu là `min_freq` thì `min_freq += 1`; `node.freq += 1`; `append` vào list mới.
- Loại: `freq_lists[min_freq].pop_oldest()`.
- `dict.setdefault(f, DLinkedList())` nghĩa là "nếu chưa có key `f` thì gán `DLinkedList()` mới, rồi trả về giá trị ứng với `f`". (Lưu ý nhỏ: `DLinkedList()` vẫn bị tạo ra mỗi lần gọi dù không dùng tới; muốn tránh có thể dùng `defaultdict(DLinkedList)` như Cách 3.)

Cell đầu tiên dưới đây cài và thử riêng danh sách liên kết đôi; cell thứ hai ghép nó thành LFU. Kết quả cuối của Example 1 phải là `{2: [4], 3: [3]}`, `min_freq = 2`, trùng với bước 10 của bảng trace mục 7.1.

In [22]:
class Node:
    __slots__ = ("key", "value", "freq", "prev", "next")    # __slots__: tiết kiệm bộ nhớ, tùy chọn

    def __init__(self, key=None, value=None):
        self.key, self.value, self.freq = key, value, 1
        self.prev = self.next = None


class DLinkedList:
    def __init__(self):
        self.head, self.tail = Node(), Node()      # 2 nút canh gác
        self.head.next, self.tail.prev = self.tail, self.head
        self.size = 0

    def append(self, node: Node) -> None:          # thêm vào cuối (mới nhất), ngay trước tail
        last = self.tail.prev
        last.next, node.prev = node, last
        node.next, self.tail.prev = self.tail, node
        self.size += 1

    def remove(self, node: Node) -> None:          # gỡ một nút đã biết, O(1)
        node.prev.next, node.next.prev = node.next, node.prev
        node.prev = node.next = None
        self.size -= 1

    def pop_oldest(self) -> Node:                  # nút ngay sau head
        node = self.head.next
        self.remove(node)
        return node

    def keys(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.key)
            cur = cur.next
        return out


dl = DLinkedList()
nodes = {k: Node(k, k * 10) for k in [1, 2, 3]}
for k in [1, 2, 3]:
    dl.append(nodes[k])
print("append 1,2,3 :", dl.keys(), "size", dl.size)
dl.remove(nodes[2])
print("remove 2     :", dl.keys())
print("pop_oldest   :", dl.pop_oldest().key, "-> còn", dl.keys())

append 1,2,3 : [1, 2, 3] size 3
remove 2     : [1, 3]
pop_oldest   : 1 -> còn [3]


In [23]:
class LFUCacheDLL:
    # Cách 4: dict key -> Node  +  dict freq -> DLinkedList  +  min_freq.

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.nodes = {}             # key -> Node
        self.freq_lists = {}        # freq -> DLinkedList
        self.min_freq = 0

    def _touch(self, node: Node) -> None:
        lst = self.freq_lists[node.freq]
        lst.remove(node)
        if lst.size == 0:
            del self.freq_lists[node.freq]
            if self.min_freq == node.freq:
                self.min_freq += 1
        node.freq += 1
        self.freq_lists.setdefault(node.freq, DLinkedList()).append(node)

    def get(self, key: int) -> int:
        node = self.nodes.get(key)
        if node is None:
            return -1
        self._touch(node)
        return node.value

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        node = self.nodes.get(key)
        if node is not None:
            node.value = value
            self._touch(node)
            return
        if len(self.nodes) == self.capacity:
            lst = self.freq_lists[self.min_freq]
            victim = lst.pop_oldest()
            if lst.size == 0:
                del self.freq_lists[self.min_freq]
            del self.nodes[victim.key]
        node = Node(key, value)
        self.nodes[key] = node
        self.freq_lists.setdefault(1, DLinkedList()).append(node)
        self.min_freq = 1


assert run_commands(LFUCacheDLL, commands_ex1, args_ex1) == expected_ex1
assert run_commands(LFUCacheDLL, commands_t2, args_t2) == expected_t2
cache = LFUCacheDLL(2)
for name, a in zip(commands_ex1[1:], args_ex1[1:]):
    getattr(cache, name)(*a)
print("Tầng cuối Example 1:", {f: lst.keys() for f, lst in sorted(cache.freq_lists.items())}, "| min_freq =", cache.min_freq)

Tầng cuối Example 1: {2: [4], 3: [3]} | min_freq = 2


## 9. Code nộp LeetCode

Bản nộp là Cách 3, gọn lại, đúng tên class `LFUCache` và đúng signature của LeetCode. Vài điểm:

- `import` đặt ở đầu file cho chắc (LeetCode thường import sẵn `collections`, nhưng tự import không bao giờ sai).
- Giữ dòng `if self.capacity == 0: return` để an toàn với bản đề cũ cho phép `capacity = 0`.
- Không `print` gì trong code nộp.

In [24]:
from collections import OrderedDict, defaultdict


class LFUCache:

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.key_to_vf = {}                              # key -> (value, freq)
        self.freq_to_keys = defaultdict(OrderedDict)     # freq -> keys, trái = lâu chưa dùng nhất
        self.min_freq = 0

    def _touch(self, key: int) -> None:
        value, f = self.key_to_vf[key]
        bucket = self.freq_to_keys[f]
        del bucket[key]
        if not bucket:
            del self.freq_to_keys[f]
            if self.min_freq == f:
                self.min_freq = f + 1
        self.freq_to_keys[f + 1][key] = None
        self.key_to_vf[key] = (value, f + 1)

    def get(self, key: int) -> int:
        if key not in self.key_to_vf:
            return -1
        self._touch(key)
        return self.key_to_vf[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.key_to_vf:
            self.key_to_vf[key] = (value, self.key_to_vf[key][1])
            self._touch(key)
            return
        if len(self.key_to_vf) == self.capacity:
            bucket = self.freq_to_keys[self.min_freq]
            victim, _ = bucket.popitem(last=False)
            if not bucket:
                del self.freq_to_keys[self.min_freq]
            del self.key_to_vf[victim]
        self.key_to_vf[key] = (value, 1)
        self.freq_to_keys[1][key] = None
        self.min_freq = 1


# Your LFUCache object will be instantiated and called as such:
# obj = LFUCache(capacity)
# param_1 = obj.get(key)
# obj.put(key,value)

In [25]:
def check(name, cls, commands, args, expected):
    actual = run_commands(cls, commands, args)
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS")


check("Example 1", LFUCache, commands_ex1, args_ex1, expected_ex1)
check("Trace 2 (min_freq nhảy, put key cũ)", LFUCache, commands_t2, args_t2, expected_t2)
check("capacity = 1", LFUCache,
      ["LFUCache", "put", "get", "put", "get", "get"], [[1], [1, 1], [1], [2, 2], [1], [2]],
      [None, None, 1, None, -1, 2])
check("get trên cache rỗng", LFUCache, ["LFUCache", "get"], [[3], [7]], [None, -1])
check("put key cũ đổi value", LFUCache,
      ["LFUCache", "put", "put", "get"], [[2], [5, 1], [5, 99], [5]], [None, None, None, 99])
check("key 0, value 10^9", LFUCache,
      ["LFUCache", "put", "get"], [[1], [0, 10**9], [0]], [None, None, 10**9])
check("capacity = 0 (bản đề cũ)", LFUCache,
      ["LFUCache", "put", "get"], [[0], [0, 0], [0]], [None, None, -1])

Example 1                          PASS
Trace 2 (min_freq nhảy, put key cũ) PASS
capacity = 1                       PASS
get trên cache rỗng                PASS
put key cũ đổi value               PASS
key 0, value 10^9                  PASS
capacity = 0 (bản đề cũ)           PASS


### 9.1 Test chéo ngẫu nhiên với trọng tài Cách 1

Ví dụ trong đề chỉ có 10 lệnh — chưa đủ tin. Ta sinh **hàng trăm** chuỗi lệnh ngẫu nhiên (seed cố định để chạy lại ra y hệt) và đòi mọi cách cho output **trùng** với `LFUCacheScan`:

- **Ít key** (`0..5`) và **capacity nhỏ** (`1..4`) để key bị loại liên tục, trùng key liên tục, hòa `freq` liên tục — đúng những chỗ dễ sai.
- Trộn `get` và `put` khoảng 50/50.

In [26]:
import random


def random_ops(n_ops, n_keys, capacity):
    commands, args = ["LFUCache"], [[capacity]]
    for _ in range(n_ops):
        k = random.randrange(n_keys)
        if random.random() < 0.5:
            commands.append("get"); args.append([k])
        else:
            commands.append("put"); args.append([k, random.randrange(100)])
    return commands, args


random.seed(460)
classes = [LFUCacheHeap, LFUCacheBuckets, LFUCacheDLL, LFUCache]
n_tests = 0
for _ in range(500):
    cmds, args = random_ops(random.randint(1, 60), random.randint(1, 6), random.randint(1, 4))
    ref = run_commands(LFUCacheScan, cmds, args)
    for cls in classes:
        assert run_commands(cls, cmds, args) == ref, (cls.__name__, cmds, args)
    n_tests += 1
# thêm vài chuỗi dài, capacity lớn hơn (để heap phải dọn rác nhiều lần)
for _ in range(20):
    cmds, args = random_ops(3000, 40, random.randint(5, 30))
    ref = run_commands(LFUCacheScan, cmds, args)
    for cls in classes:
        assert run_commands(cls, cmds, args) == ref, cls.__name__
    n_tests += 1
print(f"{n_tests} chuỗi lệnh ngẫu nhiên: cả 4 cách khớp trọng tài Cách 1. PASS")

520 chuỗi lệnh ngẫu nhiên: cả 4 cách khớp trọng tài Cách 1. PASS


### 9.2 Thử với input cỡ lớn nhất của constraints

`capacity = 10^4`, `2·10^5` lệnh, key trong `0..3·10^4` (để cache luôn đầy và phải loại thường xuyên). Cách 1 thì không dám chạy cỡ này (ước lượng `~2·10^9` phép so sánh), nên ta chạy nó ở cỡ nhỏ hơn để thấy tốc độ "phình" theo `c`.

Thời gian in ra phụ thuộc máy, chỉ nên nhìn **tỉ lệ** giữa các cách.

In [27]:
import time


def bench(cls, cmds, args):
    t0 = time.perf_counter()
    run_commands(cls, cmds, args)
    return time.perf_counter() - t0


random.seed(1)
big_cmds, big_args = random_ops(200_000, 30_000, 10_000)
for cls in [LFUCacheHeap, LFUCacheBuckets, LFUCacheDLL, LFUCache]:
    print(f"{cls.__name__:<16} c=10^4, q=2·10^5 : {bench(cls, big_cmds, big_args):.2f} s")

random.seed(2)
mid_cmds, mid_args = random_ops(20_000, 20_000, 2_000)
t_scan = bench(LFUCacheScan, mid_cmds, mid_args)
t_fast = bench(LFUCache, mid_cmds, mid_args)
print(f"\nc=2000, q=2·10^4 : Scan {t_scan:.2f} s  vs  LFUCache {t_fast:.4f} s  (chậm hơn ~{t_scan / t_fast:.0f} lần)")

LFUCacheHeap     c=10^4, q=2·10^5 : 0.27 s
LFUCacheBuckets  c=10^4, q=2·10^5 : 0.13 s


LFUCacheDLL      c=10^4, q=2·10^5 : 0.22 s
LFUCache         c=10^4, q=2·10^5 : 0.14 s



c=2000, q=2·10^4 : Scan 1.65 s  vs  LFUCache 0.0080 s  (chậm hơn ~206 lần)


## 10. Vì sao Cách 3 đúng? (invariant bằng lời)

Sau **mỗi** lệnh `get` / `put`, 4 bất biến (invariant) sau luôn đúng:

- **I1 — Mỗi key nằm đúng một tầng, đúng tầng của nó.** Key `k` có trong `key_to_vf` với `freq = f` ⇔ `k` nằm trong tầng `f` và không nằm trong tầng nào khác.
- **I2 — Trong mỗi tầng, thứ tự trái → phải là thứ tự "dùng gần nhất" tăng dần.** Lý do: một key **chỉ vào tầng `f` đúng tại lần dùng khiến nó có `freq = f`** (put mới vào tầng 1; mỗi lần dùng sau vào tầng `f + 1`), và luôn vào **cuối**. Từ lúc vào tầng đến lúc rời tầng, nó **không được dùng** (dùng là rời tầng). Vậy "thời điểm vào tầng" = "thời điểm dùng gần nhất", và vào sau thì đứng bên phải.
- **I3 — Không có tầng rỗng**, và khi cache không rỗng thì `min_freq` = `freq` nhỏ nhất trong cache.
- **I4 — `len(key_to_vf) <= capacity`.**

Kiểm tra từng thao tác giữ bất biến:

- `_touch(k)` từ `f` lên `f + 1`: I1 giữ vì ta xóa khỏi tầng `f` và thêm vào tầng `f + 1` cùng lúc sửa `freq`. I2 giữ vì `k` là key **mới dùng nhất toàn cache**, vào cuối tầng `f + 1`. I3: chỉ `freq` của `k` đổi, và chỉ có thể làm mất "tầng nhỏ nhất" nếu tầng `f` rỗng và `f == min_freq` — khi đó freq nhỏ nhất mới là `f + 1` (vì `k` ở đó, và mọi key khác có freq `> f`). Đúng Quy tắc A.
- `put` key mới khi đầy: theo I3, tầng `min_freq` chứa các key có freq nhỏ nhất; theo I2, key trái nhất là key lâu chưa dùng nhất trong số đó → **đúng key mà đề yêu cầu loại**. Sau khi chèn key mới `freq = 1` (nhỏ nhất có thể), `min_freq = 1` đúng (Quy tắc B), I4 giữ vì loại 1 rồi thêm 1.

Bốn bất biến đúng ở mọi thời điểm ⇒ mọi lần loại đều loại đúng key ⇒ mọi `get` trả đúng. Cell dưới **kiểm tra bất biến bằng code** sau từng lệnh, chạy song song với trọng tài Cách 1 (dùng `last` của trọng tài để kiểm I2).

In [28]:
def check_invariants(cache: LFUCacheBuckets, ref: LFUCacheScan) -> None:
    where = {}
    for f, bucket in cache.freq_to_keys.items():
        assert len(bucket) > 0, "I3: có tầng rỗng"
        for k in bucket:
            assert k not in where, "I1: key nằm ở 2 tầng"
            where[k] = f
    assert where == {k: vf[1] for k, vf in cache.key_to_vf.items()}, "I1: sai tầng"
    if cache.key_to_vf:
        assert cache.min_freq == min(where.values()), "I3: min_freq sai"
    for bucket in cache.freq_to_keys.values():
        lasts = [ref.data[k][2] for k in bucket]
        assert lasts == sorted(lasts), "I2: thứ tự trong tầng sai"
    assert len(cache.key_to_vf) <= cache.capacity, "I4"
    assert {k: (v[0], v[1]) for k, v in ref.data.items()} == cache.key_to_vf, "khác trọng tài"


random.seed(2024)
steps = 0
for _ in range(300):
    cmds, args = random_ops(random.randint(1, 50), random.randint(1, 7), random.randint(1, 4))
    cache, ref = LFUCacheBuckets(*args[0]), LFUCacheScan(*args[0])
    for name, a in zip(cmds[1:], args[1:]):
        assert getattr(cache, name)(*a) == getattr(ref, name)(*a)
        check_invariants(cache, ref)
        steps += 1
print(f"Đã kiểm 4 bất biến sau {steps} lệnh: PASS")

Đã kiểm 4 bất biến sau 8000 lệnh: PASS


## 11. Độ phức tạp và so sánh các cách

Cách 3 (bản nộp), với `c` = capacity:

- `get`: tra dict, `del` khỏi một `OrderedDict`, thêm vào một `OrderedDict`, sửa vài biến → **`O(1)` trung bình** (chữ "trung bình" vì dict là bảng băm).
- `put`: thêm tối đa một `popitem(last=False)` → **`O(1)` trung bình**.
- Bộ nhớ: mỗi key xuất hiện 1 lần trong `key_to_vf` và 1 lần trong đúng một tầng; số tầng ≤ số key (không có tầng rỗng) → **`O(c)`**.

| Cách | `get` | `put` | Bộ nhớ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1. dict + quét | `O(1)` | `O(c)` khi phải loại | `O(c)` | cực dễ viết, khó sai → làm trọng tài | TLE khi `c` lớn và loại nhiều |
| 2. heap + lazy deletion | `O(log c)` amortized | `O(log c)` amortized | `O(c)` (có dọn rác) | dùng công cụ tổng quát, dễ nghĩ ra | không đạt `O(1)`; logic bản ghi cũ dễ sai |
| 3. tầng `OrderedDict` + `min_freq` | `O(1)` tb | `O(1)` tb | `O(c)` | đúng yêu cầu đề, code ngắn | phải hiểu 2 quy tắc `min_freq` |
| 4. tầng DLL tự cài + `min_freq` | `O(1)` tb | `O(1)` tb | `O(c)` | không cần thư viện đặc biệt, dịch sang C++/Java dễ | dài, nhiều con trỏ dễ nối sai |

Ghi chú: tất cả đều cần `O(1)` cho "tra key có trong cache không", nhờ dict. Khác biệt nằm ở câu hỏi "key nào cần loại": Cách 1 trả lời bằng **quét**, Cách 2 bằng **heap**, Cách 3/4 bằng **cấu trúc luôn sẵn câu trả lời** (`min_freq` + đầu tầng) nhờ tính chất `freq` chỉ tăng 1.

## 12. Lỗi người mới hay gặp

**Lỗi 1 — Quên Quy tắc A (không tăng `min_freq` khi tầng nhỏ nhất rỗng).**
`min_freq` vẫn chỉ vào tầng đã bị xóa → lúc cần loại, `freq_to_keys[min_freq]` là tầng rỗng mới tạo (bẫy `defaultdict`) → `popitem` nổ `KeyError`. Nếu không xóa tầng rỗng thì còn tệ hơn: lỗi chỉ lộ ra ở những test hiếm.
*Sửa:* `if not bucket: del ...; if self.min_freq == f: self.min_freq = f + 1`.

**Lỗi 2 — Chèn key mới trước rồi mới loại.**
Sau khi chèn, `min_freq = 1` và tầng 1 có thể chỉ chứa đúng key mới → loại luôn key vừa chèn.
*Sửa:* loại **trước**, chèn **sau**.

**Lỗi 3 — `put` key đã có nhưng không tăng `freq`** (hoặc tệ hơn: đặt lại `freq = 1`).
Đề nói rõ `put` trên key có sẵn cũng là một lần dùng.
*Sửa:* cập nhật value rồi gọi `_touch(key)`.

**Lỗi 4 — Kiểm "đầy" trước khi kiểm "key đã có".**
`put` key cũ lúc cache đầy sẽ loại oan một key khác, dù số key không tăng.
*Sửa:* `if key in ...` phải đứng **trước** `if len(...) == capacity`.

**Lỗi 5 — Không phòng `capacity = 0`** (bản đề cũ): `popitem` trên tầng rỗng → `KeyError`.
*Sửa:* `if self.capacity == 0: return` đầu `put`.

**Lỗi 6 — Dùng `list` cho mỗi tầng** (`bucket.remove(key)`, `bucket.pop(0)`): đúng kết quả nhưng mỗi thao tác `O(độ dài tầng)` → với `c = 10^4` key cùng tầng 1, quay lại `O(c)` như Cách 1. *Sửa:* `OrderedDict` (hoặc dict thường — xem Bài 6) hay DLL tự cài.

**Lỗi 7 — `get` trượt mà vẫn tăng đồng hồ / bộ đếm**, hoặc trả `None` thay vì `-1`. *Sửa:* `get` trượt chỉ `return -1`.

Cell dưới cài 5 phiên bản sai đầu tiên (kế thừa `LFUCacheBuckets`, chỉ sửa đúng chỗ lỗi) và chạy trên input "bắt lỗi".

In [29]:
class WrongNoMinUpdate(LFUCacheBuckets):           # Lỗi 1
    def _touch(self, key):
        value, f = self.key_to_vf[key]
        del self.freq_to_keys[f][key]
        if not self.freq_to_keys[f]:
            del self.freq_to_keys[f]               # quên: if self.min_freq == f: self.min_freq = f + 1
        self.freq_to_keys[f + 1][key] = None
        self.key_to_vf[key] = (value, f + 1)


class WrongInsertBeforeEvict(LFUCacheBuckets):     # Lỗi 2
    def put(self, key, value):
        if key in self.key_to_vf:
            self.key_to_vf[key] = (value, self.key_to_vf[key][1]); self._touch(key); return
        self.key_to_vf[key] = (value, 1); self.freq_to_keys[1][key] = None; self.min_freq = 1
        if len(self.key_to_vf) > self.capacity:
            victim, _ = self.freq_to_keys[self.min_freq].popitem(last=False)
            del self.key_to_vf[victim]


class WrongPutNoTouch(LFUCacheBuckets):            # Lỗi 3
    def put(self, key, value):
        if key in self.key_to_vf:
            self.key_to_vf[key] = (value, self.key_to_vf[key][1]); return   # quên _touch
        super().put(key, value)


class WrongFullCheckFirst(LFUCacheBuckets):        # Lỗi 4
    def put(self, key, value):
        if len(self.key_to_vf) == self.capacity:   # kiểm đầy trước
            bucket = self.freq_to_keys[self.min_freq]
            victim, _ = bucket.popitem(last=False)
            if not bucket: del self.freq_to_keys[self.min_freq]
            del self.key_to_vf[victim]
        super().put(key, value)


class WrongNoZeroGuard(LFUCacheBuckets):           # Lỗi 5
    def put(self, key, value):
        if key in self.key_to_vf:
            self.key_to_vf[key] = (value, self.key_to_vf[key][1]); self._touch(key); return
        if len(self.key_to_vf) == self.capacity:
            victim, _ = self.freq_to_keys[self.min_freq].popitem(last=False)
            del self.key_to_vf[victim]
        self.key_to_vf[key] = (value, 1); self.freq_to_keys[1][key] = None; self.min_freq = 1


cases = [
    ("Lỗi 1", WrongNoMinUpdate, commands_ex1, args_ex1),
    ("Lỗi 2", WrongInsertBeforeEvict, ["X", "put", "get", "put", "get"], [[1], [1, 1], [1], [2, 2], [2]]),
    ("Lỗi 3", WrongPutNoTouch, ["X", "put", "put", "put", "put", "get", "get"],
     [[2], [1, 1], [2, 2], [1, 10], [3, 3], [1], [2]]),
    ("Lỗi 4", WrongFullCheckFirst, ["X", "put", "put", "put", "get"], [[2], [1, 1], [2, 2], [2, 20], [1]]),
    ("Lỗi 5", WrongNoZeroGuard, ["X", "put", "get"], [[0], [0, 0], [0]]),
]
for label, cls, cmds, args in cases:
    good = run_commands(LFUCacheScan, cmds, args)
    try:
        bad = run_commands(cls, cmds, args)
    except Exception as e:
        bad = f"{type(e).__name__}: {e}"
    print(f"{label} {cls.__name__:<22} đúng = {good}\n{'':<29}sai  = {bad}")

Lỗi 1 WrongNoMinUpdate       đúng = [None, None, None, 1, None, -1, 3, None, -1, 3, 4]
                             sai  = KeyError: 'dictionary is empty'
Lỗi 2 WrongInsertBeforeEvict đúng = [None, None, 1, None, 2]
                             sai  = [None, None, 1, None, -1]
Lỗi 3 WrongPutNoTouch        đúng = [None, None, None, None, None, 10, -1]
                             sai  = [None, None, None, None, None, -1, 2]
Lỗi 4 WrongFullCheckFirst    đúng = [None, None, None, None, 1]
                             sai  = [None, None, None, None, -1]
Lỗi 5 WrongNoZeroGuard       đúng = [None, None, -1]
                             sai  = KeyError: 'dictionary is empty'


Đọc output:

- **Lỗi 1** nổ `KeyError` ở `put(4,4)`: `min_freq` vẫn là 1 trong khi tầng 1 đã bị xóa từ bước `get(3)`.
- **Lỗi 2** (`capacity = 1`): sau `get(1)` key 1 ở tầng 2; `put(2,2)` chèn 2 vào tầng 1 rồi loại "trái nhất tầng 1" — chính là key 2. Kết quả `get(2) = -1` thay vì `2`.
- **Lỗi 3**: `put(1,10)` không tăng `freq` của 1, nên khi `put(3,3)` key 1 (cũ hơn trong tầng 1) bị loại thay cho key 2.
- **Lỗi 4**: `put(2,20)` lúc đầy loại oan key 1 → `get(1) = -1`.
- **Lỗi 5**: `capacity = 0` → `popitem` trên tầng rỗng → `KeyError`.

## 13. Mẹo nhận dạng pattern và bài liên quan

**Dấu hiệu nhận dạng "tầng theo tần suất / theo bộ đếm":**

- Đề đòi `O(1)` cho thao tác "tìm phần tử có **bộ đếm nhỏ nhất / lớn nhất**".
- Bộ đếm mỗi lần chỉ **thay đổi ±1** → không cần heap; gom phần tử theo giá trị bộ đếm thành tầng, giữ thêm một con trỏ `min` / `max`. Con trỏ chỉ dịch **một bước** vì bộ đếm chỉ đổi một bước.
- Cần phá hòa theo **thời gian** → mỗi tầng là một hàng đợi có thứ tự (`OrderedDict`, DLL, hoặc dict thường nếu chỉ thêm cuối/xóa bất kỳ/lấy đầu).

**Công thức "cache trong phỏng vấn":** `dict` để tra `O(1)` + một cấu trúc có thứ tự để biết ai bị loại. LRU = 1 hàng đợi; LFU = nhiều hàng đợi theo tần suất + `min_freq`.

**Bài liên quan:**

| Bài | Liên hệ |
|---|---|
| 146. LRU Cache (Medium) | tầng duy nhất; nền tảng của bài này (mục 2.7, Bài 4) |
| 895. Maximum Frequency Stack (Hard) | `freq → stack` + `max_freq`; `max_freq` chỉ giảm 1 bước (Bài 5) |
| 432. All O`one Data Structure (Hard) | `inc`/`dec` bộ đếm ±1, `getMaxKey`/`getMinKey` `O(1)`: tầng theo bộ đếm bằng DLL, đúng tinh thần Cách 4 |
| 1756. Design Most Recently Used Queue | hàng đợi theo độ mới, có thêm truy cập theo chỉ số |
| Thiết kế TTL cache / rate limiter (phỏng vấn system design) | cùng khung "dict + cấu trúc thứ tự + chính sách loại" |

**Một nhận xét thực tế:** LFU thuần có điểm yếu "key mới bị đè": key vừa vào có `freq = 1` nên rất dễ bị loại ngay, còn key từng "hot" ngày xưa với `freq` rất cao thì ở mãi dù không ai dùng nữa. Hệ thống thật thường dùng biến thể có **lão hóa** (aging — giảm dần bộ đếm theo thời gian) hoặc lai LRU/LFU. Bài 2 dưới đây cho bạn tự thấy hiện tượng này.

## 14. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 15) và **Đáp án** (mục 16, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

`capacity = 2`:

```text
["LFUCache","put","put","get","put","put","get","get","get"]
[[2],[1,10],[2,20],[1],[2,25],[3,30],[1],[2],[3]]
```

Viết ra list output. Câu hỏi phụ: ở `put(3,30)`, key nào bị loại và vì sao?

### Bài 2 — Tính tay bảng các tầng

`capacity = 3`:

```text
put(1,1), put(2,2), put(3,3), get(1), get(2), get(1), put(4,4), get(3), put(5,5)
```

Lập bảng giống mục 7.1 (các tầng + `min_freq` sau từng lệnh). Hai lệnh `put(4,4)` và `put(5,5)` loại key nào? Bạn nhận xét gì về số phận của key 4?

### Bài 3 — Điền TODO: tự viết lại Cách 3

Điền các chỗ `TODO` để class chạy đúng. Khi xong, bỏ comment dòng `assert` ở cuối cell để tự kiểm tra.

In [30]:
class ExerciseLFUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.key_to_vf = {}                              # key -> (value, freq)
        self.freq_to_keys = defaultdict(OrderedDict)     # freq -> keys, trái = cũ nhất
        self.min_freq = 0

    def _touch(self, key: int) -> None:
        # TODO 1: lấy (value, f) của key
        # TODO 2: xóa key khỏi tầng f
        # TODO 3: nếu tầng f rỗng -> xóa tầng; nếu min_freq == f thì min_freq = f + 1
        # TODO 4: thêm key vào cuối tầng f + 1 và cập nhật key_to_vf
        raise NotImplementedError

    def get(self, key: int) -> int:
        # TODO 5: không có -> -1; có -> _touch rồi trả value
        raise NotImplementedError

    def put(self, key: int, value: int) -> None:
        # TODO 6: capacity == 0 -> return
        # TODO 7: key đã có -> cập nhật value, _touch, return
        # TODO 8: đầy -> loại trái nhất của tầng min_freq (nhớ xóa tầng nếu rỗng)
        # TODO 9: chèn key với freq 1, min_freq = 1
        raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert run_commands(ExerciseLFUCache, commands_ex1, args_ex1) == expected_ex1
# assert run_commands(ExerciseLFUCache, commands_t2, args_t2) == expected_t2

### Bài 4 — Tự viết lại LRU Cache (#146) **chỉ bằng `dict` thường**

Không dùng `OrderedDict`. Gợi ý từ mục 2.2: `dict` giữ thứ tự chèn, `pop` rồi chèn lại để đưa key về cuối, `next(iter(d))` để lấy key cũ nhất. Mọi thao tác phải `O(1)` trung bình.

In [31]:
class ExerciseLRUCacheDict:
    def __init__(self, capacity: int):
        # TODO: self.capacity, self.data = {}
        pass

    def get(self, key: int) -> int:
        # TODO: không có -> -1; có -> pop rồi chèn lại (về cuối), trả value
        raise NotImplementedError

    def put(self, key: int, value: int) -> None:
        # TODO: có rồi -> pop; chưa có mà đầy -> xóa next(iter(self.data)); cuối cùng chèn key
        raise NotImplementedError


# assert run_commands(ExerciseLRUCacheDict, lru_cmds, lru_args) == [None, None, None, 1, None, -1, None, -1, 3, 4]

### Bài 5 — Biến thể: LeetCode 895 — Maximum Frequency Stack (Hard)

Thiết kế `FreqStack`:

- `push(val)`: đẩy `val` vào stack.
- `pop()`: lấy ra và trả về phần tử **xuất hiện nhiều nhất** trong stack; nếu hòa, lấy phần tử **gần đỉnh stack nhất** (được đẩy vào gần đây nhất).

```text
["FreqStack","push","push","push","push","push","push","pop","pop","pop","pop"]
[[],[5],[7],[5],[7],[4],[5],[],[],[],[]]
Output: [null,null,null,null,null,null,null,5,7,5,4]
```

Yêu cầu `O(1)` mỗi thao tác. Hãy chuyển giao ý tưởng "tầng theo tần suất + con trỏ cực trị" của bài LFU.

In [32]:
class ExerciseFreqStack:
    def __init__(self):
        # TODO: freq (val -> số lần), group (freq -> list), max_freq
        pass

    def push(self, val: int) -> None:
        raise NotImplementedError

    def pop(self) -> int:
        raise NotImplementedError


# fs_cmds = ["FreqStack","push","push","push","push","push","push","pop","pop","pop","pop"]
# fs_args = [[],[5],[7],[5],[7],[4],[5],[],[],[],[]]
# assert run_commands(ExerciseFreqStack, fs_cmds, fs_args) == [None]*7 + [5, 7, 5, 4]

### Bài 6 — Mini challenge: LFU chỉ bằng `dict` thường + nhật ký loại bỏ

Viết `ExerciseLFUCacheLog`:

1. Mỗi tầng là một **`dict` thường** (không `OrderedDict`) — vẫn `O(1)` trung bình.
2. Có thêm thuộc tính `self.evicted`: list các key bị loại, theo thứ tự.

Kiểm tra: chạy Example 1 phải ra đúng output và `evicted == [2, 1]`.

In [33]:
class ExerciseLFUCacheLog:
    def __init__(self, capacity: int):
        # TODO
        pass

    def get(self, key: int) -> int:
        raise NotImplementedError

    def put(self, key: int, value: int) -> None:
        raise NotImplementedError


# c = ExerciseLFUCacheLog(2)
# out = [None] + [getattr(c, n)(*a) for n, a in zip(commands_ex1[1:], args_ex1[1:])]
# assert out == expected_ex1 and c.evicted == [2, 1]

## 15. Gợi ý theo mức

**Bài 1.** (1) Nhớ `put(2,25)` là một lần **dùng** key 2. (2) Trước `put(3,30)`, cả 1 và 2 đều có `freq = 2` — ai vào tầng 2 trước? (3) Key 1 vào tầng 2 ở `get(1)`, key 2 vào ở `put(2,25)`.

**Bài 2.** (1) Vẽ 3 tầng, mỗi lệnh chỉ dịch 1 key. (2) Sau `get(1), get(2), get(1)`: 1 ở tầng 3, 2 ở tầng 2, 3 vẫn ở tầng 1. (3) Key mới luôn vào tầng 1 với `min_freq = 1` → nó là ứng viên bị loại kế tiếp.

**Bài 3.** (1) Chép lại đúng 5 bước của `_touch` ở mục 5.1. (2) Nhớ thứ tự trong `put`: `capacity == 0` → key cũ → đầy → chèn. (3) `victim, _ = bucket.popitem(last=False)`; `if not bucket: del self.freq_to_keys[self.min_freq]`.

**Bài 4.** (1) "Đưa về cuối" = xóa rồi chèn lại. (2) `value = self.data.pop(key); self.data[key] = value`. (3) Trong `put`: `if key in self.data: self.data.pop(key)` / `elif len(self.data) == self.capacity: del self.data[next(iter(self.data))]` / `self.data[key] = value`.

**Bài 5.** (1) Giống LFU nhưng ngược: cần **max** tần suất, và hòa thì lấy **mới nhất** (stack). (2) `group[f]` là list các giá trị theo thứ tự lần đẩy thứ `f` của chúng; một giá trị xuất hiện 3 lần nằm trong cả `group[1]`, `group[2]`, `group[3]`. (3) `pop`: `val = group[max_freq].pop()`, `freq[val] -= 1`, nếu `group[max_freq]` rỗng thì `max_freq -= 1` — chỉ lùi **một bước**, giống Quy tắc A của LFU.

**Bài 6.** (1) Tầng là dict `{key: None}`: thêm cuối `bucket[key] = None`, xóa bất kỳ `del bucket[key]`, lấy cũ nhất `next(iter(bucket))`. (2) Khi loại: `victim = next(iter(bucket)); del bucket[victim]; self.evicted.append(victim)`. (3) Phần còn lại y hệt Cách 3.

## 16. Đáp án tham khảo

**Bài 1.** Output: `[null, null, null, 10, null, null, -1, 25, 30]`.

- `put(1,10)`, `put(2,20)`: tầng 1 = `[1, 2]`.
- `get(1) = 10`: tầng 1 = `[2]`, tầng 2 = `[1]`.
- `put(2,25)`: key cũ → value 25, key 2 lên tầng 2 **sau** key 1: tầng 2 = `[1, 2]`; tầng 1 rỗng → `min_freq = 2`.
- `put(3,30)`: đầy → loại trái nhất tầng 2 = **key 1** (cùng `freq = 2` với key 2 nhưng key 1 dùng lần cuối sớm hơn). Nhiều bạn nghĩ loại key 2 vì "quên" `put(2,25)` là một lần dùng.
- `get(1) = -1`, `get(2) = 25` (value mới), `get(3) = 30`.

**Bài 2.**

| Lệnh | Các tầng sau lệnh | `min_freq` | Ghi chú |
|---|---|:---:|---|
| `put(1,1)`, `put(2,2)`, `put(3,3)` | `f1:[1,2,3]` | 1 | |
| `get(1)` | `f1:[2,3] f2:[1]` | 1 | |
| `get(2)` | `f1:[3] f2:[1,2]` | 1 | |
| `get(1)` | `f1:[3] f2:[2] f3:[1]` | 1 | |
| `put(4,4)` | `f1:[4] f2:[2] f3:[1]` | 1 | loại **3** |
| `get(3)` | (không đổi) | 1 | trả -1 |
| `put(5,5)` | `f1:[5] f2:[2] f3:[1]` | 1 | loại **4** |

Key 4 vừa vào đã bị loại ở lệnh `put` kế tiếp, vì key mới luôn có `freq = 1` thấp nhất — đúng điểm yếu "key mới bị đè" của LFU thuần đã nói ở mục 13. Cell dưới kiểm chứng Bài 1 và Bài 2 bằng code.

In [34]:
# Kiểm chứng Bài 1
b1_cmds = ["LFUCache", "put", "put", "get", "put", "put", "get", "get", "get"]
b1_args = [[2], [1, 10], [2, 20], [1], [2, 25], [3, 30], [1], [2], [3]]
assert run_commands(LFUCacheScan, b1_cmds, b1_args) == [None, None, None, 10, None, None, -1, 25, 30]
assert run_commands(LFUCache, b1_cmds, b1_args) == [None, None, None, 10, None, None, -1, 25, 30]

# Kiểm chứng Bài 2
b2_cmds = ["LFUCache", "put", "put", "put", "get", "get", "get", "put", "get", "put"]
b2_args = [[3], [1, 1], [2, 2], [3, 3], [1], [2], [1], [4, 4], [3], [5, 5]]
trace_lfu(b2_cmds, b2_args)

LFUCache(3)
 1. put(1, 1)  -> None | 1:(1, 1)                   | f1:[1]                 | min=1 
 2. put(2, 2)  -> None | 1:(1, 1) 2:(2, 1)          | f1:[1, 2]              | min=1 
 3. put(3, 3)  -> None | 1:(1, 1) 2:(2, 1) 3:(3, 1) | f1:[1, 2, 3]           | min=1 
 4. get(1)     -> 1    | 1:(1, 2) 2:(2, 1) 3:(3, 1) | f1:[2, 3]  f2:[1]      | min=1 
 5. get(2)     -> 2    | 1:(1, 2) 2:(2, 2) 3:(3, 1) | f1:[3]  f2:[1, 2]      | min=1 
 6. get(1)     -> 1    | 1:(1, 3) 2:(2, 2) 3:(3, 1) | f1:[3]  f2:[2]  f3:[1] | min=1 
 7. put(4, 4)  -> None | 1:(1, 3) 2:(2, 2) 4:(4, 1) | f1:[4]  f2:[2]  f3:[1] | min=1 loại 3
 8. get(3)     -> -1   | 1:(1, 3) 2:(2, 2) 4:(4, 1) | f1:[4]  f2:[2]  f3:[1] | min=1 
 9. put(5, 5)  -> None | 1:(1, 3) 2:(2, 2) 5:(5, 1) | f1:[5]  f2:[2]  f3:[1] | min=1 loại 4


In [35]:
# Đáp án Bài 3
class AnswerLFUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.key_to_vf = {}
        self.freq_to_keys = defaultdict(OrderedDict)
        self.min_freq = 0

    def _touch(self, key: int) -> None:
        value, f = self.key_to_vf[key]                   # TODO 1
        bucket = self.freq_to_keys[f]
        del bucket[key]                                  # TODO 2
        if not bucket:                                   # TODO 3
            del self.freq_to_keys[f]
            if self.min_freq == f:
                self.min_freq = f + 1
        self.freq_to_keys[f + 1][key] = None             # TODO 4
        self.key_to_vf[key] = (value, f + 1)

    def get(self, key: int) -> int:
        if key not in self.key_to_vf:                    # TODO 5
            return -1
        self._touch(key)
        return self.key_to_vf[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:                           # TODO 6
            return
        if key in self.key_to_vf:                        # TODO 7
            self.key_to_vf[key] = (value, self.key_to_vf[key][1])
            self._touch(key)
            return
        if len(self.key_to_vf) == self.capacity:         # TODO 8
            bucket = self.freq_to_keys[self.min_freq]
            victim, _ = bucket.popitem(last=False)
            if not bucket:
                del self.freq_to_keys[self.min_freq]
            del self.key_to_vf[victim]
        self.key_to_vf[key] = (value, 1)                 # TODO 9
        self.freq_to_keys[1][key] = None
        self.min_freq = 1


assert run_commands(AnswerLFUCache, commands_ex1, args_ex1) == expected_ex1
assert run_commands(AnswerLFUCache, commands_t2, args_t2) == expected_t2
print("Bài 3 PASS")

Bài 3 PASS


In [36]:
# Đáp án Bài 4 — LRU chỉ bằng dict thường
class AnswerLRUCacheDict:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data = {}                       # thứ tự chèn: đầu = lâu chưa dùng nhất

    def get(self, key: int) -> int:
        if key not in self.data:
            return -1
        value = self.data.pop(key)           # đưa về cuối = xóa rồi chèn lại
        self.data[key] = value
        return value

    def put(self, key: int, value: int) -> None:
        if key in self.data:
            self.data.pop(key)
        elif len(self.data) == self.capacity:
            del self.data[next(iter(self.data))]     # key cũ nhất
        self.data[key] = value


assert run_commands(AnswerLRUCacheDict, lru_cmds, lru_args) == [None, None, None, 1, None, -1, None, -1, 3, 4]
random.seed(146)
for _ in range(300):
    cmds, args = random_ops(random.randint(1, 50), random.randint(1, 6), random.randint(1, 4))
    assert run_commands(AnswerLRUCacheDict, cmds, args) == run_commands(LRUCache, cmds, args)
print("Bài 4 PASS (khớp LRUCache bằng OrderedDict trên 300 chuỗi ngẫu nhiên)")

Bài 4 PASS (khớp LRUCache bằng OrderedDict trên 300 chuỗi ngẫu nhiên)


In [37]:
# Đáp án Bài 5 — LeetCode 895 Maximum Frequency Stack
class AnswerFreqStack:
    def __init__(self):
        self.freq = {}                       # val -> số lần đang có trong stack
        self.group = defaultdict(list)       # f -> các val theo thứ tự lần đẩy thứ f
        self.max_freq = 0

    def push(self, val: int) -> None:
        f = self.freq.get(val, 0) + 1
        self.freq[val] = f
        self.max_freq = max(self.max_freq, f)
        self.group[f].append(val)

    def pop(self) -> int:
        val = self.group[self.max_freq].pop()        # nhiều nhất, hòa thì mới nhất
        self.freq[val] -= 1
        if not self.group[self.max_freq]:
            self.max_freq -= 1                       # chỉ lùi 1 bước (giống Quy tắc A)
        return val


fs_cmds = ["FreqStack", "push", "push", "push", "push", "push", "push", "pop", "pop", "pop", "pop"]
fs_args = [[], [5], [7], [5], [7], [4], [5], [], [], [], []]
assert run_commands(AnswerFreqStack, fs_cmds, fs_args) == [None] * 7 + [5, 7, 5, 4]


def freq_stack_brute(pushes, n_pop):         # trọng tài: quét toàn bộ stack
    st, out = list(pushes), []
    for _ in range(n_pop):
        cnt = {}
        for v in st:
            cnt[v] = cnt.get(v, 0) + 1
        best = max(cnt.values())
        i = max(j for j, v in enumerate(st) if cnt[v] == best)   # gần đỉnh nhất
        out.append(st.pop(i))
    return out


random.seed(895)
for _ in range(300):
    pushes = [random.randint(1, 4) for _ in range(random.randint(1, 15))]
    k = random.randint(1, len(pushes))
    fs = AnswerFreqStack()
    for v in pushes:
        fs.push(v)
    assert [fs.pop() for _ in range(k)] == freq_stack_brute(pushes, k)
print("Bài 5 PASS")

Bài 5 PASS


In [38]:
# Đáp án Bài 6 — LFU với tầng là dict thường + nhật ký loại
class AnswerLFUCacheLog:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.key_to_vf = {}
        self.freq_to_keys = {}               # freq -> dict{key: None}, thứ tự chèn = thứ tự vào tầng
        self.min_freq = 0
        self.evicted = []

    def _touch(self, key: int) -> None:
        value, f = self.key_to_vf[key]
        bucket = self.freq_to_keys[f]
        del bucket[key]
        if not bucket:
            del self.freq_to_keys[f]
            if self.min_freq == f:
                self.min_freq = f + 1
        self.freq_to_keys.setdefault(f + 1, {})[key] = None
        self.key_to_vf[key] = (value, f + 1)

    def get(self, key: int) -> int:
        if key not in self.key_to_vf:
            return -1
        self._touch(key)
        return self.key_to_vf[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.key_to_vf:
            self.key_to_vf[key] = (value, self.key_to_vf[key][1])
            self._touch(key)
            return
        if len(self.key_to_vf) == self.capacity:
            bucket = self.freq_to_keys[self.min_freq]
            victim = next(iter(bucket))      # key vào tầng sớm nhất
            del bucket[victim]
            if not bucket:
                del self.freq_to_keys[self.min_freq]
            del self.key_to_vf[victim]
            self.evicted.append(victim)
        self.key_to_vf[key] = (value, 1)
        self.freq_to_keys.setdefault(1, {})[key] = None
        self.min_freq = 1


c = AnswerLFUCacheLog(2)
out = [None] + [getattr(c, n)(*a) for n, a in zip(commands_ex1[1:], args_ex1[1:])]
assert out == expected_ex1 and c.evicted == [2, 1]
print("Example 1 evicted =", c.evicted)

random.seed(6)
for _ in range(300):
    cmds, args = random_ops(random.randint(1, 60), random.randint(1, 6), random.randint(1, 4))
    assert run_commands(AnswerLFUCacheLog, cmds, args) == run_commands(LFUCacheScan, cmds, args)
print("Bài 6 PASS (khớp trọng tài trên 300 chuỗi ngẫu nhiên)")

Example 1 evicted = [2, 1]
Bài 6 PASS (khớp trọng tài trên 300 chuỗi ngẫu nhiên)


## 17. Template ngắn nên nhớ và checklist trước khi submit

### Template LFU (`O(1)`)

```python
from collections import OrderedDict, defaultdict

class LFUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.kv = {}                          # key -> (value, freq)
        self.fk = defaultdict(OrderedDict)    # freq -> keys (trái = cũ nhất)
        self.minf = 0

    def _touch(self, k):                      # dịch k: tầng f -> tầng f+1
        v, f = self.kv[k]
        del self.fk[f][k]
        if not self.fk[f]:
            del self.fk[f]
            if self.minf == f: self.minf = f + 1
        self.fk[f + 1][k] = None
        self.kv[k] = (v, f + 1)

    def get(self, k):
        if k not in self.kv: return -1
        self._touch(k); return self.kv[k][0]

    def put(self, k, v):
        if self.cap == 0: return
        if k in self.kv:
            self.kv[k] = (v, self.kv[k][1]); self._touch(k); return
        if len(self.kv) == self.cap:          # loại TRƯỚC
            victim, _ = self.fk[self.minf].popitem(last=False)
            if not self.fk[self.minf]: del self.fk[self.minf]
            del self.kv[victim]
        self.kv[k] = (v, 1); self.fk[1][k] = None; self.minf = 1   # chèn SAU
```

**Câu thần chú:** *"Dùng key → dịch lên một tầng; tầng cũ rỗng mà là min thì min + 1. Key mới → đầy thì loại trái nhất tầng min, rồi chèn vào tầng 1, min = 1."*

### Checklist trước khi submit

- [ ] Tên class `LFUCache`, method `get(self, key) -> int`, `put(self, key, value) -> None` đúng như LeetCode.
- [ ] `get` trượt trả `-1`, **không** thay đổi trạng thái.
- [ ] `put` key **đã có**: cập nhật value **và** tăng `freq` (gọi `_touch`), **không** loại ai.
- [ ] `put` key mới: kiểm "key đã có" **trước** kiểm "đầy"; **loại trước, chèn sau**; chèn xong đặt `min_freq = 1`.
- [ ] `_touch`: tầng rỗng → xóa tầng, và chỉ khi `f == min_freq` mới `min_freq = f + 1`.
- [ ] Hòa `freq` → loại **trái nhất** (lâu chưa dùng nhất) của tầng: `popitem(last=False)`, không phải `popitem()`.
- [ ] Không đọc `defaultdict[f]` chỉ để kiểm tra (sẽ tạo tầng rỗng) — dùng `in`.
- [ ] Phòng thủ `capacity == 0` (bản đề cũ).
- [ ] Đã test Example 1, `capacity = 1`, put key cũ khi đầy, và test chéo ngẫu nhiên với bản quét `O(c)`.